# 00 — Master Editability: editable causal structure in trained world models (GRU + refined RSSM)

**Purpose — the single source of truth for this pillar.** This notebook is where we consolidate *how we
investigate **editable causal structure** in trained world models*: the language, the metrics, and the
probes we use to ask whether a learned hidden state is a **canonical, factored, predictively-sufficient**
carrier of the world's degrees of freedom — measured the **same way across architectures** (GRU and
refined RSSM here; every figure/table is built to hold an Nth model with no re-layout). It is deliberately
**provisional**: a *proposal* for language/metrics that may later be folded into `pim`, **not** the
codebase itself. Do not over-index on any single number or naming choice.

**How to read this notebook (synthesis-tier conventions).** The *invariant spine* — definitions, metric
formulas, the pipeline — is stable and lives in the section headers and the **definitions table** (right
after the bootstrap). Every *result that moves as models/experiments evolve* lives in a clearly-marked
**`Current results (updated YYYY-MM-DD)`** block inside its section — never woven into a header, a
definition, or a figure title. Cheap artifacts (states, probes, edits, waterfalls) are recomputed here;
expensive ones (intrinsic dim / curvature on the 200k-state bank) are **cited** with provenance.

Structure — one idea per section (**headline + figure + table**), every claim citable as "cell [N] / Fig K":
- **§0 Premise** — what the *physical* minimal statistic is, and why the *causal/belief* state the model needs is legitimately larger.
- **§1 Geometry** — intrinsic dimension, linear-hull dimension, and curvature of the visited-state manifold.
- **§2 Recoverability** — can `(pos,vel)` be read out of a single `h`? linear vs MLP; is velocity temporal or instantaneous?
- **§3 Canonicality / fiber** — is `h` a *function* of `(pos,vel)`, or does an off-`(pos,vel)` fiber remain? GRU vs RSSM det-core.
- **§4 Editing head-to-head (the core comparison)** — do latent edits that hit the *readout* also move the *observations*? five editors on both models, per-model waterfalls + metrics vs the sim's true post-edit trajectory.
- **§5 Synthesis** — ties the sections to the organizing hypothesis (editability ⟺ canonical, factored, predictively-sufficient state), as *hypothesis*.

**Models / data (checkpoints — keep this list current as the thread evolves):**
- GRU — `runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt` (H=256)
- refined RSSM — `runs/rssm/4_dset4_refined_best/best_model.pt` (`model.sample=False`; flat = cat[det 256, stoch 64] = 320)
- data — `datasets/4_fixed_refl_inview`, `n_obj_keep=2`, teacher-forced **test** split.

**Source notebooks (sources of truth):** `canonical_state_editing`, `geodesic_walk_k150`,
`manifold_geometry_diagnostic`, `diagnostic_corrections`, `../rssm_structure/rssm_state_geometry`.
Corrected numbers: `research/scratch/2026-07-08-diagnostic-corrections.md` + `candidate-*.md`.

> **Caveat stated once (applies throughout):** all probes are **in-sample fit** (fit and evaluated on the
> same masked entries, no held-out split). Absolute R²/residual magnitudes are therefore optimistic; the
> **comparisons** (linear-vs-MLP, single-vs-2-frame, GRU-vs-RSSM, det-vs-s, editor-vs-editor) are the
> load-bearing quantities and are unaffected.


---
## Bootstrap — load BOTH models, teacher-force, velocities, shared themes & helpers

In [ ]:
# [1] Shared bootstrap: imports, config, load GRU + refined RSSM, teacher-force, velocities.
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display
import h5py

import pim.eval as eval
from pim.extractors import LinearExtractor, MLPExtractor, StateDefinition, identity_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer,
)
from pim.editors.manifold_steering import _pca_subspace
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 512
NUM_WORKERS = 6
N_OBJ       = 2
DATA_DIR    = "../../../datasets/4_fixed_refl_inview"
OUT = "/tmp/master_editability"; os.makedirs(OUT, exist_ok=True)

GRU_CKPT  = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
RSSM_CKPT = "../../../runs/rssm/4_dset4_refined_best/best_model.pt"

# ---- Data (shared) ----
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
DT = float(test.config["dataset"]["sim"]["dt"])

# ---- GRU ----
gru, gru_info = load_checkpoint(GRU_CKPT, device=DEVICE)
H_GRU = gru.hidden_size
preds_gru, states_gru = eval.teacher_force(gru, test_loader, device=DEVICE)   # (N,39,256)

# ---- Refined RSSM (deterministic prior-mean; posterior-mean states) ----
rssm, rssm_info = load_checkpoint(RSSM_CKPT, device=DEVICE)
rssm.sample = False
DET, STO = rssm.cfg.det_size, rssm.cfg.stoch_size
H_RSSM = rssm.hidden_size
preds_rssm, states_rssm = eval.teacher_force(rssm, test_loader, device=DEVICE)  # (N,39,320)
assert DET == 256 and STO == 64 and H_RSSM == 320, (DET, STO, H_RSSM)

print(f"device={DEVICE}  dt={DT}")
print(f"GRU  : {gru_info.run_name} (ep {gru_info.epoch}, val_loss={gru_info.val_loss:.5f})  H={H_GRU}  states={states_gru.shape}")
print(f"RSSM : {rssm_info.run_name} (ep {rssm_info.epoch}, val_loss={rssm_info.val_loss:.5f})  H={H_RSSM} (det={DET},stoch={STO})  states={states_rssm.shape}")

In [ ]:
# [2] Velocities from HDF5 `velocities` (aligned like positions[:, :-1]); flat (pos,vel) targets; visibility.
v_test = h5py.File(test.h5_path, "r")["velocities"][:, :, :N_OBJ, :].astype(np.float32)   # (N,40,2,2)
vel_tf  = v_test[:, :-1, :, :]                        # (N,39,2,2) aligned with states
pos_tf  = test.positions[:, :-1, :N_OBJ, :]          # (N,39,2,2)
vis_tf  = test.is_visible[:, :-1, :N_OBJ].all(axis=2)  # (N,39) both objects visible

print("velocity temporal std (constant-vel sim -> ~0):", float(v_test.std(axis=1).mean()))
print("mean|v| =", float(np.abs(vel_tf).mean()), " (tiny -> depresses absolute vel R², relative story robust)")

posflat_tf = pos_tf.reshape(*pos_tf.shape[:2], N_OBJ*2)               # (N,39,4) [x0,y0,x1,y1]
velflat_tf = vel_tf.reshape(*vel_tf.shape[:2], N_OBJ*2)               # (N,39,4) [vx0,vy0,vx1,vy1]
posvel_tf  = np.concatenate([posflat_tf, velflat_tf], -1)            # (N,39,8)
LATE_T = 15                                                          # late-t = t>=15 (velocity well-determined)

In [ ]:
# [3] Shared THEMES (light for metrics, dark for simulator/waterfall) + generic probe/g-fit helpers.
OK = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","pink":"#CC79A7","yellow":"#E69F00","grey":"#999999"}
def style_ax(ax):
    ax.spines[["top","right"]].set_visible(False); ax.grid(alpha=0.25, lw=0.6)
plt.style.use("default")

# --- generic MLP/linear regressor: feats -> target, returns (pred, R2_overall, R2_percomp, resid_frac) ---
def _fit_regress(X, Y, kind, hidden=256, n_epochs=100, lr=2e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    Din, Dout = X.shape[1], Y.shape[1]
    Xt = torch.from_numpy(X.astype(np.float32)).to(DEVICE); Yt = torch.from_numpy(Y.astype(np.float32)).to(DEVICE)
    if kind == "linear":
        Xa = torch.cat([Xt, torch.ones(Xt.shape[0],1,device=DEVICE)],1)
        sol = torch.linalg.lstsq(Xa, Yt).solution
        with torch.no_grad(): pred = Xa @ sol
    else:
        net = nn.Sequential(nn.Linear(Din,hidden), nn.ReLU(), nn.Linear(hidden,hidden), nn.ReLU(),
                            nn.Linear(hidden,Dout)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=lr); bs = 4096; Nn = Xt.shape[0]
        for ep in range(n_epochs):
            perm = torch.randperm(Nn, device=DEVICE)
            for i in range(0, Nn, bs):
                idx = perm[i:i+bs]
                loss = ((net(Xt[idx]) - Yt[idx])**2).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): pred = net(Xt)
    resid2 = ((pred - Yt)**2).sum(0)
    tot2   = ((Yt - Yt.mean(0,keepdim=True))**2).sum(0)
    r2_pc  = (1 - resid2/torch.clamp(tot2,min=1e-12)).cpu().numpy()
    r2_all = float(1 - resid2.sum()/tot2.sum())
    resid_frac = float((( (pred-Yt)**2).sum() / (Yt**2).sum()).sqrt())    # ||Y-pred||/||Y||
    return pred.cpu().numpy(), r2_all, r2_pc, resid_frac

def fit_probe(feats_tf, y_tf, mask, kind, **kw):
    """feats_tf:(N,T,F) y_tf:(N,T,D) mask:(N,T)bool. Returns dict on masked entries."""
    X = feats_tf[mask]; Y = y_tf[mask]
    pred, r2, r2pc, rfrac = _fit_regress(X, Y, kind, **kw)
    return dict(r2=r2, r2pc=r2pc, resid_frac=rfrac, n=X.shape[0])

print("themes + helpers ready")

---
## Definitions & metrics (the invariant spine — read once)

Every non-obvious term and metric, with its formula, units, and better-direction. Results that *move*
are NOT here — they live in each section's `Current results (updated …)` block. (RMSE throughout, never MSE.)

| symbol / metric | definition & formula | units | better |
|---|---|---|---|
| `(pos, vel)` | per-object position & velocity; the **physical minimal sufficient statistic** = 2 obj × (2 pos + 2 vel) = **8-dim** | — | — |
| `h` | model hidden state under test: GRU `h ∈ ℝ²⁵⁶`; RSSM flat = cat[det 256, stoch 64] = `ℝ³²⁰` | — | — |
| causal / belief state | predictively-sufficient statistic under noise: a **posterior over `(pos,vel)`** (Kalman / ε-machine causal state); legitimately **≥ 8-dim** (see §0) | — | — |
| early-t frames | frames with time index **t < 15**. The recursive filter / belief has **not yet converged** from the short noisy history, so velocity is still **underdetermined** and harder to read out of a single frame | frame index | — |
| late-t frames | frames with time index **t ≥ 15** (threshold `LATE_T = 15`). The filter / belief has **converged**, so velocity is **well-determined** and cleanly readable from a single frame | frame index | — |
| PCA hull dim @p% | smallest #PCA components of the visited-`h` bank with cumulative variance ≥ p%; **linear-hull upper bound** on dimension | dims | — |
| intrinsic dim (TwoNN) | model-free; `d = 1 / mean(log(r₂/r₁))`, `r₁,r₂` = 1st/2nd nearest-neighbour distances (Facco 2017); no linearity assumed | dims | — |
| intrinsic dim (MLE) | model-free; Levina–Bickel MLE over k=20 NN, bias-corrected `× (k−2)/(k−1)`; no linearity assumed | dims | — |
| tangent rotation | mean principal angle between local-PCA **tangent subspaces** (k=64 NN, top-8 comps) of nearby states; 0° = flat | deg | ↓ flatter |
| position / velocity R² | `1 − SS_resid/SS_tot` of a probe `h → (pos or vel)`; **linear** (lstsq) or **MLP** | — | ↑ |
| single-frame vs two-frame probe | **single-frame** reads velocity from one hidden state `h_t`; **two-frame** reads it from the pair `[h_{t-1}, h_t]`. two-frame ≈ single-frame ⇒ velocity is present *instantaneously*, not built from a temporal difference | — | — |
| differenced (dh) probe | reads velocity from `dh = h_t − h_{t-1}`; a temporal-difference feature (tested as a control) | — | — |
| fiber residual | `‖h − g(pos,vel)‖ / ‖h‖`, `g` an MLP; fraction of `h` **not** a function of the 8-dim statistic (0 = fully canonical) | frac of ‖h‖ | ↓ |
| readout RMSE (§4) | `√mean((A·h_edit + b − target_pos)²)` — probe readout of the edited state vs the post-edit target positions, **at the edit step before any rollout** | position | ↓ |
| GT next-step RMSE (§4) | `RMSE(generated obs at rollout step 1, sim clean obs at frame ef+1)`; observation-space accuracy one step after the edit | obs intensity | ↓ |
| per-step GT-trajectory RMSE (§4) | `RMSE(generated obs at step s, sim clean obs at frame ef+s)` — distance to the **time-evolving** true post-edit trajectory | obs intensity | ↓ |
| step-0 →static-target RMSE (§4) | `RMSE(generated obs at step 0, static render of the edit-frame target positions)`; **step-0 direct-edit check only** — objects keep moving, so later steps must not be compared to a static render | obs intensity | ↓ |
| obs-change / % of swap (§4) | `RMSE(obs_edit, obs_unsteered)` at step 0; reported as % of the **true-state-swap** obs-change (the proper 100% — what swapping in the teacher-forced post-edit state changes) | obs intensity / % | → 100 |
| ghost-ray ratio (§4) | **ghost rays** := rays where the edited object was pre-edit AND must not be post-edit (`pre_render_id==edit_obj & tgt_render_id!=edit_obj`); ratio = mean step-0 intensity on ghost rays, editor ÷ unsteered | ratio | ↓ (1 = ghost remains) |
| persistence (§4) | per-step GT-trajectory RMSE as a function of rollout step; a rise = the edit does not persist. Whether a failing edit *reverts to the unsteered rollout* or *collapses off-distribution* is read from the per-step distance to the unsteered rollout plus the waterfalls | obs vs step | — |
| global-PCA hull residual (§4) | raw `‖h − proj_global(h)‖` onto the global 90%-var PCA subspace; reference = the same residual for real states (printed per model) | ‖h‖ units | ↓ |
| leave-out local-PCA residual (§4) | `‖q − proj_local(q)‖ / ‖q − local mean‖`, local PCA on the k=64 nearest bank states **excluding the query's own nearest neighbour** (without the exclusion a state already in the bank projects onto itself — a tautology); reference = real states, printed beside every use | fraction | ↓ |


---
## §0 — Premise: the world's *physical* minimal statistic is `(pos, vel)` = 8-dim

**Headline.** The simulator is exactly **constant-velocity** (`pos_{t+1} = pos_t + vel·dt`, dt=1; velocity
never changes). So the *entire* future is determined by the current `(positions, velocities)` — an
**8-dimensional** minimal sufficient statistic for 2 objects. Everything downstream asks whether the
learned hidden state `h` is a *canonical, factored, editable* carrier of that statistic, or a
predictively-sufficient tangle around it.

> **Conceptual note — physical statistic vs causal/belief state (PROVISIONAL; framing under refinement by Sevan).**
> `(pos,vel)` = 8-dim is the **generative** minimal sufficient statistic: it determines the world's future
> exactly. But it is **not** what the model can condition on. With **process noise** (position σ≈0.04) and
> **observation noise** (σ≈0.2), `(pos,vel)` is generally **not confidently inferable from a finite noisy
> history** — so the *predictively*-sufficient statistic is a **belief** (posterior) over `(pos,vel)`: a
> recursive-Bayesian / **Kalman-filter** state carrying a mean **and its uncertainty**. In
> **computational-mechanics** terms (Crutchfield / Shalizi **ε-machine**), the **causal state** is the
> minimal partition of pasts that is predictively sufficient; under noise that partition is this belief,
> and the statistical complexity `Cμ` **exceeds** the entropy of the physical state. The belief collapses
> toward ~8 dims only at the filter's **steady state**. **Implication (state plainly):** a hidden state
> storing **> 8 dims is *expected*, not pathological** — so "`h` is non-canonical vs the 8-dim *physical*
> state" (§3) is **not** the same claim as "`h` is non-canonical vs the *causal / belief* state." §1–§3
> measure against the 8-dim **physical** statistic; read them with this distinction in mind.

> **Current results (updated 2026-07-14).** velocity temporal-std ≈ 1.3e-8 over a trajectory ⇒ the sim is
> constant-velocity to numerical precision, so the physical minimal statistic is `(pos,vel)` = 8-dim.
> Carriers under test: GRU `h ∈ ℝ²⁵⁶`, RSSM flat `∈ ℝ³²⁰` (det 256 + stoch 64).


In [ ]:
# [4] §0 — verify constant-velocity (temporal std of velocity ~0), state the DOF budget, draw Fig 0 pipeline.
vstd = float(v_test.std(axis=1).mean())
print("="*72)
print("§0 PREMISE — constant-velocity sim → (pos,vel) is the physical sufficient statistic")
print("="*72)
print(f"velocity temporal std over a trajectory : {vstd:.3e}   (≈0 → velocity is constant)")
print(f"physical minimal sufficient statistic   : (pos,vel) = 2 obj x (2 pos + 2 vel) = {N_OBJ*4}-dim")
print(f"learned carriers under test             : GRU h in R^{H_GRU} ; RSSM flat in R^{H_RSSM} (det {DET} + stoch {STO})")
print("Per the conceptual note above: under process+obs noise the CAUSAL/BELIEF state is legitimately >= 8-dim")
print("(a posterior over (pos,vel)), so storing > 8 dims is EXPECTED. §1-§3 measure against the 8-dim physical state.")

# ---- Fig 0 — architecture-agnostic pipeline: world -> render -> observe -> [world model] -> h -> 3 probes ----
# Light/analysis theme (a framing schematic, not simulator output). Non-overlapping boxes; no colinear pass-through.
fig, ax = plt.subplots(figsize=(11.5, 5.2)); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

def _box(x, y, w, h, text, fc, ec="0.25", fs=10, alpha=0.9, lw=1.4, weight="normal", tc="k"):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec, alpha=alpha, lw=lw, zorder=1))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs, zorder=3, weight=weight, color=tc)

def _harrow(x0, x1, y, color="0.25", lw=1.9):
    ax.annotate("", xy=(x1, y), xytext=(x0, y), arrowprops=dict(arrowstyle="-|>", lw=lw, color=color), zorder=2)

def _varrow(x, y0, y1, color="0.25", lw=1.9):
    ax.annotate("", xy=(x, y1), xytext=(x, y0), arrowprops=dict(arrowstyle="-|>", lw=lw, color=color), zorder=2)

TY, TH, TMID = 0.66, 0.20, 0.76        # top pipeline row: y0, height, mid-y (arrow height)
# pipeline boxes (neutral fills; model colours are reserved for the world-model SLOTS)
_box(0.010, TY, 0.150, TH, "World state\n(pos, vel)\n8-dim", OK["green"], fs=9.5, alpha=0.30)
_box(0.205, TY, 0.125, TH, "Renderer", "#c9ced6", fs=10, alpha=0.9)
_box(0.375, TY, 0.155, TH, "1D observations\n(range scans)", OK["yellow"], fs=9.5, alpha=0.30)
_box(0.575, TY, 0.225, TH, "", "#eef1f5", fs=10, alpha=0.95)   # world-model container (label + slots below)
_box(0.845, TY, 0.140, TH, "hidden state\nh", "#cfe0ee", fs=11, alpha=0.95, weight="bold")
for x0, x1 in [(0.160, 0.205), (0.330, 0.375), (0.530, 0.575), (0.800, 0.845)]:
    _harrow(x0, x1, TMID)
# world-model container: title + three architecture SLOTS (visibly holds multiple architectures)
ax.text(0.6875, 0.812, "World Model", ha="center", va="center", fontsize=10.5, weight="bold", zorder=3)
for sx, lab, col in [(0.588, "GRU", OK["blue"]), (0.658, "RSSM", OK["orange"]), (0.728, "…", OK["grey"])]:
    _box(sx, 0.675, 0.062, 0.085, lab, col, fs=8.5, alpha=0.35, ec="0.35")

# hidden state -> distribution rail -> three labelled probes off h
RAILY = 0.50; HX = 0.915
_varrow(HX, TY, RAILY)
ax.plot([0.16, HX], [RAILY, RAILY], color="0.25", lw=1.6, zorder=2)
ax.text(0.30, 0.53, "three probes off h", fontsize=8.5, style="italic", color="0.35")
probes = [
    (0.16, OK["pink"],   "Recoverability\n\nh → (pos, vel)\n\ncan the 8-dim statistic\nbe read out?  (§2)"),
    (0.50, "#2AA198",    "Editability\n\nΔh → rollout → obs\n\ndoes the edit move\nthe observation?  (§4)"),
    (0.84, "#8172B3",    "Geometry\n\nmanifold of visited h\n\nintrinsic dim / curvature\n(§1)"),
]
for cx, col, txt in probes:
    _varrow(cx, RAILY, 0.34)
    _box(cx - 0.14, 0.10, 0.28, 0.24, txt, col, fs=9, alpha=0.20, ec="0.35")

ax.text(0.5, 0.965,
        "Fig 0 — architecture-agnostic pipeline: world → render → observe → world model → hidden state h, probed three ways",
        ha="center", va="center", fontsize=11)
fig.savefig(f"{OUT}/fig0_premise.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §1 — Geometry: how many DOF does the visited-state manifold have, and is it flat or curved?

**What this section measures.** Three reads on the geometry of the bank of visited hidden states `h`:
(1) **linear-hull dimension** — #PCA components for 90% of state variance (an *upper* bound: a curved
low-dim surface needs more linear dims than its intrinsic dim); (2) **intrinsic dimension** — model-free
estimators (**TwoNN**, **MLE**) that assume no linearity; (3) **curvature** — how far the local tangent
plane rotates between nearby states (large rotation ⇒ curved). All three are read against the physical
**8 DOF**. *Curvature is the geometric reason linear / min-norm probe edits leave the manifold* (→ §4).

*Recompute here (cheap):* PCA scree **and TwoNN/MLE intrinsic dim for every model**, each on its own
state bank. *Cite (expensive, 200k-bank):* tangent-rotation angles from `manifold_geometry_diagnostic`
(GRU) and `rssm_state_geometry` (RSSM).

> **Current results (updated 2026-07-14).** Model-free intrinsic dim **brackets the 8 physical DOF for the GRU** (the RSSM's ~10 sits somewhat *above* it) and both sit
> **far below** the linear hull ⇒ a strongly curved embedding; the curved-embedding pattern replicates on
> the RSSM, though its intrinsic dim is higher.
> Intrinsic dim (own-bank): **TwoNN GRU 5.2 / RSSM 9.6**, **MLE GRU 6.9 / RSSM 10.0**.
> Linear hull @90%: **GRU 38 / RSSM 35**. Tangent rotation @ NN spacing (cited): **≈56° (GRU) /
> ≈65° (RSSM)**. Per-model bars in Fig 1b; full table (with "how computed") in cell [5].


In [ ]:
# [5] §1 — recompute PCA scree + model-free intrinsic dim (TwoNN, MLE) for EVERY model on its own bank.
from IPython.display import Markdown

def scree(states, H):
    bank = states.reshape(-1, H)
    sub = _pca_subspace(torch.from_numpy(bank).float().to(DEVICE), n_components=H, var_threshold=1.0)
    ratio = sub.explained_variance_ratio.cpu().numpy(); cum = np.cumsum(ratio)
    dims = {p: int((cum < p).sum()) + 1 for p in (0.70, 0.90, 0.95)}
    return ratio, cum, dims

ratio_g, cum_g, dims_g = scree(states_gru, H_GRU)
ratio_r, cum_r, dims_r = scree(states_rssm, H_RSSM)

# ---- model-free intrinsic-dim estimators (method mirrors manifold_geometry_diagnostic cell [5]) ----
# chunked kNN so the (sample x bank) distance matrix stays memory-bounded.
@torch.no_grad()
def _knn_dists(Q, X, k, chunk=2000):
    out = []
    for i in range(0, Q.shape[0], chunk):
        d = torch.cdist(Q[i:i+chunk], X)                       # (chunk, |X|)
        vals, _ = torch.topk(d, k + 1, largest=False, dim=1)   # ascending, incl self at col 0
        out.append(vals)
    return torch.cat(out, 0)

@torch.no_grad()
def two_nn_id(X, sample=20000, seed=0):
    """TwoNN (Facco 2017): d = 1 / mean(log(r2/r1)); r1,r2 = 1st/2nd NN distances. Model-free."""
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    vals = _knn_dists(X[idx], X, 2)                            # [self~0, r1, r2]
    r1, r2 = vals[:, 1], vals[:, 2]
    keep = (r1 > 1e-9) & (r2 > r1)
    logmu = torch.log(r2[keep] / r1[keep])
    return float(1.0 / logmu.mean())

@torch.no_grad()
def mle_id(X, k=20, sample=20000, seed=0):
    """Levina-Bickel MLE over k NN, bias-corrected x (k-2)/(k-1). Model-free."""
    g = torch.Generator(device=X.device).manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g, device=X.device)[:min(sample, X.shape[0])]
    vals = _knn_dists(X[idx], X, k)                            # (n, k+1) incl self
    Tk = vals[:, 1:k+1].clamp_min(1e-9); logT = torch.log(Tk)
    m_inv = (logT[:, k-1:k] - logT[:, :k-1]).mean(1)
    mk = 1.0 / m_inv.clamp_min(1e-9)
    return float(mk.mean() * (k - 2) / (k - 1))

def id_bank(states, H, cap=200_000, seed=0):
    bank = states.reshape(-1, H)
    rng = np.random.RandomState(seed)
    idx = rng.choice(bank.shape[0], size=min(cap, bank.shape[0]), replace=False)
    return torch.from_numpy(bank[idx]).float().to(DEVICE)

bank_g = id_bank(states_gru, H_GRU); bank_r = id_bank(states_rssm, H_RSSM)
twonn_g, mle_g = two_nn_id(bank_g), mle_id(bank_g, k=20)
twonn_r, mle_r = two_nn_id(bank_r), mle_id(bank_r, k=20)
del bank_g, bank_r
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# ---- cited (expensive 200k-bank) tangent-rotation curvature; GRU intrinsic kept for a provenance check ----
CITED = dict(
    physical_dof=8,
    tangent_angle_gru=56.0, tangent_angle_rssm=65.2,             # mean tangent principal angle @ NN spacing
    intrinsic_twonn_gru_cited=5.2, intrinsic_mle_gru_cited=6.9,  # manifold_geometry_diagnostic (GRU, 200k bank)
    dims90_gru_cited=38, dims90_rssm_cited=34,
    honest_local_resid_real="0.75-0.84", global_flat_resid_real=1.75)
TANGENT_METHOD = ("mean principal angle between local-PCA tangent subspaces (k=64 NN, top-8 comps) of "
                  "nearby states, over 80 anchors x 60 stratified targets")
PHYS_DOF = CITED["physical_dof"]

# ---- per-model geometry record: built to hold N models (no two-model hardcoding) ----
MODELS_GEO = {
    "GRU":  dict(color=OK["blue"],   H=H_GRU,  cum=cum_g, dims=dims_g, twonn=twonn_g, mle=mle_g, tangent=CITED["tangent_angle_gru"]),
    "RSSM": dict(color=OK["orange"], H=H_RSSM, cum=cum_r, dims=dims_r, twonn=twonn_r, mle=mle_r, tangent=CITED["tangent_angle_rssm"]),
}

# ---- metrics table WITH a "how computed / meaning" column (rendered markdown; no pandas dependency) ----
rows = [
    ("PCA hull dim @70%",                dims_g[0.70], dims_r[0.70],
     "# PCA comps for >=70% of state variance (linear-hull; upper bound on dim)"),
    ("PCA hull dim @90%",                dims_g[0.90], dims_r[0.90],
     "# PCA comps for >=90% var; a curved low-dim surface needs MORE linear dims than its intrinsic dim"),
    ("PCA hull dim @95%",                dims_g[0.95], dims_r[0.95],
     "# PCA comps for >=95% var"),
    ("intrinsic dim (TwoNN)",            round(twonn_g, 2), round(twonn_r, 2),
     "model-free; d=1/mean(log(r2/r1)), r1,r2=1st/2nd NN dist (Facco 2017); no linearity assumed"),
    ("intrinsic dim (MLE k=20)",         round(mle_g, 2), round(mle_r, 2),
     "model-free; Levina-Bickel MLE over 20 NN, bias-corrected x(k-2)/(k-1)"),
    ("physical DOF",                     PHYS_DOF, PHYS_DOF,
     "(pos,vel) x 2 obj = 8; the generative minimal statistic (reference)"),
    ("tangent rotation @NN (deg, cited)", CITED["tangent_angle_gru"], CITED["tangent_angle_rssm"],
     TANGENT_METHOD + "; higher = more curved [cited: manifold_geometry_diagnostic (GRU) / rssm_state_geometry (RSSM)]"),
]
print("=== §1 GEOMETRY TABLE (scree + TwoNN/MLE recomputed per model; tangent rotation cited) ===")
_md = "| metric | GRU | RSSM | how computed / meaning |\n|---|---|---|---|\n" + \
      "".join(f"| {m} | {g} | {r} | {h} |\n" for (m, g, r, h) in rows)
display(Markdown(_md))
# plain-text mirror (agent-readable / no-render)
print(f"{'metric':30s} {'GRU':>8s} {'RSSM':>8s}")
for (m, g, r, _h) in rows:
    print(f"{m:30s} {str(g):>8s} {str(r):>8s}")

print(f"\nGRU intrinsic-dim provenance check: TwoNN here {twonn_g:.2f} vs cited {CITED['intrinsic_twonn_gru_cited']}; "
      f"MLE here {mle_g:.2f} vs cited {CITED['intrinsic_mle_gru_cited']} (manifold_geometry_diagnostic, 200k bank).")
print(f"Read: model-free intrinsic dim sits close to the {PHYS_DOF} physical DOF (GRU slightly below, RSSM above) "
      f"and far below the linear hull (@90%: GRU {dims_g[0.90]} / RSSM {dims_r[0.90]}) → strongly curved embedding, both models.")
print(f"MARKDOWN_NUMS vstd={vstd:.3e} twonn_g={twonn_g:.2f} mle_g={mle_g:.2f} twonn_r={twonn_r:.2f} "
      f"mle_r={mle_r:.2f} hull90_g={dims_g[0.90]} hull90_r={dims_r[0.90]} hull95_g={dims_g[0.95]} hull95_r={dims_r[0.95]}")

In [ ]:
# [6] Fig 1 — geometry (built to hold N models): (a) PCA scree, (b) intrinsic-dim estimators grouped by model, (c) curvature.
model_names = list(MODELS_GEO)                      # ["GRU", "RSSM", ... Nth]
mcolor = {m: MODELS_GEO[m]["color"] for m in model_names}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) PCA scree — cumulative variance per model; hull@90% as dashed vlines (NOT hard-coded in the title)
ax = axes[0]
for m in model_names:
    cum = MODELS_GEO[m]["cum"]
    ax.plot(np.arange(1, len(cum) + 1), cum, color=mcolor[m], lw=2, label=m)
    ax.axvline(MODELS_GEO[m]["dims"][0.90], color=mcolor[m], ls="--", lw=1)
ax.axhline(0.90, color="0.6", ls=":", lw=1)
ax.set_xlim(0, 80); ax.set_xlabel("# PCA components"); ax.set_ylabel("cumulative variance")
ax.set_title("(a) PCA scree — cumulative variance (dashed = hull@90% per model)")
ax.legend(fontsize=8); style_ax(ax)

# (b) intrinsic-dim estimators: x = estimator category, ONE color-coded bar per model side-by-side, 8-DOF line
ax = axes[1]
cats = ["TwoNN", "MLE", "hull@90%"]
def _catvals(m):
    g = MODELS_GEO[m]; return [g["twonn"], g["mle"], g["dims"][0.90]]
x = np.arange(len(cats)); nM = len(model_names); w = 0.8 / nM
for j, m in enumerate(model_names):
    off = (j - (nM - 1) / 2) * w
    vals = _catvals(m)
    ax.bar(x + off, vals, width=w, color=mcolor[m], alpha=0.9, label=m)
    for xi, v in zip(x + off, vals):
        ax.text(xi, v + 0.6, f"{v:.1f}", ha="center", fontsize=7.5)
ax.axhline(PHYS_DOF, color="k", ls="--", lw=1.5, label=f"physical {PHYS_DOF} DOF")
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel("dimension"); ax.set_title("(b) intrinsic dim (TwoNN, MLE) vs linear hull@90%, per model")
ax.legend(fontsize=8); style_ax(ax)

# (c) tangent-rotation curvature — one bar per model (cited)
ax = axes[2]
tvals = [MODELS_GEO[m]["tangent"] for m in model_names]
ax.bar(model_names, tvals, color=[mcolor[m] for m in model_names], alpha=0.9)
for xi, v in enumerate(tvals):
    ax.text(xi, v + 1, f"{v:.0f}°", ha="center", fontsize=8)
ax.axhline(30, color="0.6", ls=":", lw=1, label="30° curvature scale")
ax.set_ylim(0, 80); ax.set_ylabel("tangent rotation @ NN spacing (deg)")
ax.set_title("(c) local tangent reorients strongly (cited)"); ax.legend(fontsize=8); style_ax(ax)

fig.suptitle("Fig 1 — State geometry: low intrinsic dim, strongly curved, fat linear hull (built to hold N models)",
             y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_geometry.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)


---
## §2 — Recoverability: can `(pos, vel)` be read out of a single hidden state `h`?

**What this section measures.** Given one hidden state `h_t`, can a probe recover the physical statistic?
Two probes are compared — a **linear** least-squares map and a 2-layer **MLP** — on two targets:
- **Position** (`h_t → pos`), linear vs MLP. For the RSSM we additionally split the state into its
  **deterministic core `h_det` (256)**, its **stochastic part `s` (64)**, and the **full 320**, to ask where
  position is stored.
- **Velocity** (`→ vel`), as a 2×2 of {linear, MLP} × {single-frame `h_t`, two-frame `[h_{t-1}, h_t]`}, plus a
  differenced control `dh = h_t − h_{t-1}`. Comparing single-frame vs two-frame tells us whether velocity is
  present *instantaneously* or must be built from a temporal difference; comparing linear vs MLP tells us
  whether the readout is linear or curved.

> **Note — why we split frames into early-t and late-t (not all frames pooled).** Because the world is noisy
> (process noise on position, observation noise), the model runs an implicit recursive filter; its belief over
> `(pos, vel)` only sharpens as evidence accumulates. So we report two regimes separately (defined in the
> table above): **early-t = frames t < 15** (belief not yet converged ⇒ velocity underdetermined) and
> **late-t = frames t ≥ 15** (converged ⇒ velocity well-determined). Pooling all frames blends the two and
> understates how cleanly velocity is readable once the filter has settled. The full single-frame-vs-two-frame
> 2×2 for **both** regimes is kept in the printed table (cell [7]); the bars (Fig 2) show single-frame only.

> **Current results (updated 2026-07-15).** **Position** is nearly linear: linear R² ≈ 0.84, MLP ≈ 0.97 (GRU;
> RSSM similar at 0.85 / 0.96); it lives in the RSSM's **deterministic core** (linear R²: det-only 0.84 ≈ full
> 0.85, both above stochastic-`s`-only 0.58). **Velocity** is readable from a **single frame but only
> nonlinearly**: single-frame linear R² ≈ 0.57–0.59, single-frame MLP ≈ 0.93–0.94 (both models, late-t). A
> two-frame window adds essentially nothing over single-frame MLP (two-frame − single-frame ≈ 0: ≤ 0.007 late-t,
> ≤ 0.02 early-t, and slightly negative for the RSSM), and the differenced `dh` control is strictly worse
> (0.66–0.81 vs 0.87–0.95 single-frame MLP). Velocity readout is markedly cleaner **late-t** (single-frame MLP
> ≈ 0.93–0.94) than **early-t** (≈ 0.75–0.87), consistent with a filter that has converged. **Implication:** the
> old "velocity is *temporal* (0.47 linear → 0.76 two-frame MLP)" reading was a **linear-vs-MLP confound and is
> retired** — velocity is **nonlinear-instantaneous, not temporal**. See cell [7]/[8] tables and Fig 2.

In [ ]:
# [7] §2 — velocity readout 2x2 {linear,MLP} x {single-frame, two-frame}, per model, early-t vs late-t; + dh control.
from IPython.display import Markdown
VCOMP = ["vx0","vy0","vx1","vy1"]

def build_feats(states, regime="late"):
    """regime in {'early','late'}: early = frames t<15 (filter not converged), late = t>=15 (converged)."""
    sf  = states[:, 1:, :]                                            # single-frame h_t
    win = np.concatenate([states[:, :-1, :], states[:, 1:, :]], -1)   # two-frame [h_{t-1}, h_t]
    dh  = states[:, 1:, :] - states[:, :-1, :]                        # differenced control
    y   = velflat_tf[:, 1:, :]
    mask = vis_tf[:, 1:] & vis_tf[:, :-1]                             # both objects visible in both frames
    rm = np.zeros_like(mask)
    if regime == "early":
        rm[:, :LATE_T-1] = True                                      # aligned index t-1 < 14  <=>  t < 15
    elif regime == "late":
        rm[:, LATE_T-1:] = True                                      # t >= 15
    else:
        raise ValueError(regime)
    return sf, win, dh, y, (mask & rm)

def run_vel_2x2(states, regime):
    sf, win, dh, y, mask = build_feats(states, regime)
    return {
        ("sf","lin"):  fit_probe(sf,  y, mask, "linear"),
        ("sf","mlp"):  fit_probe(sf,  y, mask, "mlp"),
        ("win","lin"): fit_probe(win, y, mask, "linear"),
        ("win","mlp"): fit_probe(win, y, mask, "mlp"),
        ("dh","mlp"):  fit_probe(dh,  y, mask, "mlp"),
    }

vel_res = {}
for lbl, st in [("GRU", states_gru), ("RSSM", states_rssm)]:
    for reg in ["early", "late"]:
        vel_res[(lbl, reg)] = run_vel_2x2(st, reg)

# ---- clearly-demarcated table (rendered markdown; matches the §1 cell [5] style, no pandas dependency) ----
reg_label = {"early": "early-t (t<15)", "late": "late-t (t≥15)"}
VCOLS = ["single-frame linear", "single-frame MLP", "two-frame linear", "two-frame MLP",
         "differenced dh (MLP)", "two-frame − single-frame (MLP)"]
def _vrow(lbl, reg):
    o = vel_res[(lbl, reg)]
    return [o[("sf","lin")]["r2"], o[("sf","mlp")]["r2"], o[("win","lin")]["r2"],
            o[("win","mlp")]["r2"], o[("dh","mlp")]["r2"], o[("win","mlp")]["r2"] - o[("sf","mlp")]["r2"]]

print("Velocity readout R² (overall) — can a probe read velocity out of the hidden state? "
      "Higher is better; single-frame vs two-frame tells instantaneous vs temporal.")
_md = "| model | frame regime | " + " | ".join(VCOLS) + " |\n"
_md += "|---|---|" + "|".join(["---"] * len(VCOLS)) + "|\n"
for lbl in ["GRU", "RSSM"]:
    for reg in ["early", "late"]:
        _md += f"| {lbl} | {reg_label[reg]} | " + " | ".join(f"{v:.3f}" for v in _vrow(lbl, reg)) + " |\n"
display(Markdown(_md))
# plain-text mirror (agent-readable / no-render)
print(f"{'model / regime':24s} {'sf-lin':>7s} {'sf-MLP':>7s} {'2f-lin':>7s} {'2f-MLP':>7s} {'dh-MLP':>7s} {'2f−sf':>7s}")
for lbl in ["GRU", "RSSM"]:
    for reg in ["early", "late"]:
        print(f"{lbl+' '+reg_label[reg]:24s} " + " ".join(f"{v:7.3f}" for v in _vrow(lbl, reg)))
print("Read: single-frame MLP ≈ two-frame MLP (last column ≈ 0) ⇒ velocity is present instantaneously, "
      "not built from a temporal difference; linear ≪ MLP ⇒ the readout is nonlinear.")

In [ ]:
# [8] §2 — position recoverability (linear vs MLP) + RSSM h_det / s / full position split.
pos_res = {}
for lbl, st in [("GRU", states_gru), ("RSSM", states_rssm)]:
    pos_res[(lbl, "lin")] = fit_probe(st, posflat_tf, vis_tf, "linear")
    pos_res[(lbl, "mlp")] = fit_probe(st, posflat_tf, vis_tf, "mlp")

h_det   = states_rssm[..., :DET]; s_stoch = states_rssm[..., DET:]
pos_split = {
    "RSSM full (320)":    fit_probe(states_rssm, posflat_tf, vis_tf, "linear")["r2"],
    "RSSM h_det (256)":   fit_probe(h_det,       posflat_tf, vis_tf, "linear")["r2"],
    "RSSM s_stoch (64)":  fit_probe(s_stoch,     posflat_tf, vis_tf, "linear")["r2"],
}

# ---- table 1 (rendered markdown): position readout R^2, linear vs MLP, per model ----
print("Position readout R² (overall) — how well can position be read out of a single hidden state? Higher is better.")
_md = "| model | linear probe R² | MLP probe R² |\n|---|---|---|\n"
for m in ["GRU", "RSSM"]:
    _md += f"| {m} | {pos_res[(m,'lin')]['r2']:.3f} | {pos_res[(m,'mlp')]['r2']:.3f} |\n"
display(Markdown(_md))

# ---- table 2 (rendered markdown): RSSM position split — where does position live? ----
print("Where does RSSM position live? Linear-probe R² on the deterministic core, the stochastic part, and the full state.")
_md2 = "| RSSM state block | linear probe R² |\n|---|---|\n"
for k, v in pos_split.items():
    _md2 += f"| {k} | {v:.3f} |\n"
display(Markdown(_md2))
print("Read: det-core R² ≈ full-state R² and both ≫ stochastic-s R² ⇒ position lives in the deterministic core.")

In [ ]:
# [9] Fig 2 — recoverability of (pos,vel) from a single hidden state h.
#   (a) velocity from single-frame h_t: linear vs MLP, GRU/RSSM x early-t/late-t
#   (b) per-component velocity from single-frame h_t (late-t, MLP), GRU vs RSSM
#   (c) position: linear vs MLP, with the RSSM det/s/full split
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

# (a) single-frame velocity readout: two probe types x (model, regime) bars
ax = axes[0]
cats = ["linear probe", "MLP probe"]
series = [("GRU","early",OK["blue"],0.45), ("GRU","late",OK["blue"],1.0),
          ("RSSM","early",OK["orange"],0.45), ("RSSM","late",OK["orange"],1.0)]
x = np.arange(len(cats)); w = 0.8 / len(series)
for j,(lbl,reg,c,alpha) in enumerate(series):
    off = (j - (len(series)-1)/2) * w
    vals = [vel_res[(lbl,reg)][("sf","lin")]["r2"], vel_res[(lbl,reg)][("sf","mlp")]["r2"]]
    ax.bar(x + off, vals, w, color=c, alpha=alpha, label=f"{lbl} {reg}-t")
    for xi, v in zip(x + off, vals):
        ax.text(xi, v + 0.015, f"{v:.2f}", ha="center", fontsize=6.5)
ax.axhline(1, color="0.6", ls=":", lw=1)
ax.set_xticks(x); ax.set_xticklabels(cats); ax.set_ylim(0, 1.1)
ax.set_ylabel("velocity R² (overall)")
ax.set_title("(a) velocity from a single frame h_t")
ax.legend(fontsize=6, ncol=2); style_ax(ax)

# (b) per-component velocity readout, single-frame MLP, late-t, GRU vs RSSM
ax = axes[1]
x = np.arange(4); w = 0.38
for j,(lbl,c) in enumerate([("GRU",OK["blue"]),("RSSM",OK["orange"])]):
    pc = vel_res[(lbl,"late")][("sf","mlp")]["r2pc"]
    ax.bar(x + (j-0.5)*w, pc, w, color=c, label=lbl)
ax.set_xticks(x); ax.set_xticklabels(VCOMP); ax.set_ylim(0, 1.02)
ax.set_ylabel("velocity R² per component")
ax.set_title("(b) per-component velocity, single-frame MLP (late-t)")
ax.legend(fontsize=7); style_ax(ax)

# (c) position readout: linear vs MLP, with RSSM det/s/full split annotated
ax = axes[2]
x = np.arange(2); w = 0.35
ax.bar(x-w/2, [pos_res[("GRU","lin")]["r2"], pos_res[("RSSM","lin")]["r2"]], w, color=OK["blue"], label="linear")
ax.bar(x+w/2, [pos_res[("GRU","mlp")]["r2"], pos_res[("RSSM","mlp")]["r2"]], w, color=OK["orange"], label="MLP")
ax.set_xticks(x); ax.set_xticklabels(["GRU","RSSM"]); ax.set_ylabel("position R²"); ax.set_ylim(0, 1.02)
txt = "RSSM position (linear):\n full {full:.2f} | det {det:.2f} | s {s:.2f}".format(
    full=pos_split["RSSM full (320)"], det=pos_split["RSSM h_det (256)"], s=pos_split["RSSM s_stoch (64)"])
ax.text(0.98, 0.05, txt, transform=ax.transAxes, ha="right", va="bottom", fontsize=7, bbox=dict(fc="0.95", ec="0.7"))
ax.set_title("(c) position: linear vs MLP (with RSSM det/s split)")
ax.legend(fontsize=8); style_ax(ax)

fig.suptitle("Fig 2 — Recoverability of (pos, vel) from a single hidden state h", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_recoverability.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §3 — Canonicality / fiber-collapse: is the hidden state a *function* of `(pos, vel)`?

**What this section measures.** A hidden state is **canonical** with respect to the physical statistic if it is
a *function* of `(pos, vel)` — every state with the same `(pos, vel)` maps to the same `h`. We test this by
fitting the best map `g(pos, vel) → block` (a **linear** map and an **MLP**, `g` fit on the 8-dim statistic)
and measuring the **fiber residual** `‖block − g(pos,vel)‖ / ‖block‖`: the fraction of the block that **no**
function of `(pos, vel)` can explain (0 = fully canonical; large = an off-`(pos,vel)` fiber remains, carrying
history / scaffolding beyond the minimal statistic). We run it on the **GRU `h`** and, for the RSSM, on the
**full 320**, the **deterministic core `h_det` (256)**, and the **stochastic part `s` (64)** separately, so the
stochastic latent does not inflate the deterministic core's number. A large linear→MLP drop signals a **curved**
embedding. (Recall the §0 caveat: this is measured against the 8-dim *physical* statistic, not the belief state.)

> **Current results (updated 2026-07-15).** The GRU `h` leaves an MLP fiber residual of **≈ 0.34** — about a
> third of `h` is **not** a function of `(pos, vel)`, so `h` is **non-canonical** against the physical statistic.
> The RSSM's deterministic core is **no more canonical**: det-only residual **≈ 0.37 ≈ GRU** (do not over-read
> the ~0.03 gap). The full-320 residual (**≈ 0.60**) is **inflated by the stochastic part `s`** (its own residual
> **≈ 0.89** — legitimately not a function of `(pos, vel)`, as expected for a KL-regularised latent), so read the
> **det-core** number for the architecture comparison. **Implication:** the KL structure buys **no** extra
> canonicity — the deterministic cores of both architectures are equally non-canonical. See cell [10] table and Fig 3.

In [ ]:
# [10] §3 — fiber-collapse residual: fit g(pos,vel)->block (linear & MLP) for GRU h and RSSM full/det/s.
def fit_g(block, kind):
    X = posvel_tf[vis_tf]; Y = block[vis_tf]
    _, r2, _, rfrac = _fit_regress(X, Y, kind, hidden=512, n_epochs=120, lr=1.5e-3)
    return rfrac, r2

blocks = {"GRU h (256)": states_gru, "RSSM full (320)": states_rssm,
          "RSSM h_det (256)": h_det, "RSSM s_stoch (64)": s_stoch}
fiber = {}
fiber_rows = []
for name, blk in blocks.items():
    lrf, lr2 = fit_g(blk, "linear"); mrf, mr2 = fit_g(blk, "mlp")
    fiber[name] = dict(lin=(lrf, lr2), mlp=(mrf, mr2))
    fiber_rows.append((name, lrf, lr2, mrf, mr2))

# ---- clearly-demarcated table (rendered markdown): is each block a function of the 8-dim (pos, vel)? ----
print("Fiber-collapse residual — is each hidden-state block a function of the 8-dim (pos, vel)? "
      "Residual fraction ‖block − g(pos,vel)‖ / ‖block‖; lower means more canonical (0 = fully a function of it).")
_md = ("| block | linear g: residual fraction | linear g: R² on block | "
       "MLP g: residual fraction | MLP g: R² on block |\n|---|---|---|---|---|\n")
for name, lrf, lr2, mrf, mr2 in fiber_rows:
    _md += f"| {name} | {lrf:.3f} | {lr2:.3f} | {mrf:.3f} | {mr2:.3f} |\n"
display(Markdown(_md))
# plain-text mirror (agent-readable / no-render)
print(f"{'block':22s} {'lin resid':>10s} {'lin R²':>8s} {'MLP resid':>10s} {'MLP R²':>8s}")
for name, lrf, lr2, mrf, mr2 in fiber_rows:
    print(f"{name:22s} {lrf:10.3f} {lr2:8.3f} {mrf:10.3f} {mr2:8.3f}")

# vars reused by Fig 3 (reference line / annotation)
gru_r  = fiber["GRU h (256)"]["mlp"][0];   det_r = fiber["RSSM h_det (256)"]["mlp"][0]
full_r = fiber["RSSM full (320)"]["mlp"][0]; s_r  = fiber["RSSM s_stoch (64)"]["mlp"][0]
print(f"Read: GRU h MLP residual {gru_r:.3f} ≈ RSSM det-core {det_r:.3f} (equally non-canonical); "
      f"the stochastic part inflates the full state (full {full_r:.3f} vs det {det_r:.3f}; s alone {s_r:.3f}).")

In [ ]:
# [11] Fig 3 — fiber residual bars: (a) residual fraction linear vs MLP, (b) R2 on block.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
order = ["GRU h (256)","RSSM full (320)","RSSM h_det (256)","RSSM s_stoch (64)"]
x = np.arange(len(order)); w = 0.38

# (a) residual fraction: lower = more nearly a function of (pos,vel); labels on BOTH linear and MLP bars
ax = axes[0]
lin_rf = [fiber[n]["lin"][0] for n in order]; mlp_rf = [fiber[n]["mlp"][0] for n in order]
ax.bar(x-w/2, lin_rf, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_rf, w, label="MLP g",    color=OK["orange"])
for xi, v in zip(x-w/2, lin_rf): ax.text(xi, v+0.012, f"{v:.3f}", ha="center", fontsize=7.5, color=OK["blue"])
for xi, v in zip(x+w/2, mlp_rf): ax.text(xi, v+0.012, f"{v:.3f}", ha="center", fontsize=7.5, color=OK["orange"])
ax.axhline(gru_r, color=OK["green"], ls=":", lw=1.2, label=f"GRU h reference ({gru_r:.3f})")
ax.set_xticks(x); ax.set_xticklabels(order, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("residual fraction ‖blk − g‖ / ‖blk‖"); ax.set_ylim(0, 1.15)   # headroom so legend clears the bars
ax.set_title("(a) is the block a function of (pos, vel)?")
ax.legend(fontsize=7, loc="upper left"); style_ax(ax)

# (b) variance of each block explained by g(pos,vel)
ax = axes[1]
lin_r2 = [fiber[n]["lin"][1] for n in order]; mlp_r2 = [fiber[n]["mlp"][1] for n in order]
ax.bar(x-w/2, lin_r2, w, label="linear g", color=OK["blue"])
ax.bar(x+w/2, mlp_r2, w, label="MLP g",    color=OK["orange"])
ax.set_xticks(x); ax.set_xticklabels(order, rotation=18, ha="right", fontsize=8)
ax.set_ylabel("R² on block"); ax.set_ylim(0, 1.0)
ax.set_title("(b) variance of block explained by g(pos, vel)")
ax.legend(fontsize=7); style_ax(ax)

fig.suptitle("Fig 3 — Fiber collapse: is each hidden-state block a function of the 8-dim (pos, vel)?", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_fiber.png", dpi=130, bbox_inches="tight"); display(fig); plt.close(fig)

---
## §4 — Editing head-to-head (the core comparison)

All editors run on the **same** edit set for **both world models** (§4a GRU, §4b RSSM; §4c+ reserved for
future architectures): teacher-force each edits-split sequence to `edit_frame = 20` on the **pre-edit**
observations, apply the editor to the hidden state `h`, then roll the model out freely for 15 steps.
The intended outcome is the **true post-edit trajectory** — the simulation's clean observations from the
edit frame on (`edits.clean_obs`, teleport applied). That simulation trajectory is the GT reference in
every figure and table; it is never a model output. Data: `datasets/4_fixed_refl_inview` edits split,
first 64 samples; models/checkpoints as in cell [1].

**References (not editors):**

| reference | what it is |
|---|---|
| **GT (sim)** | the simulation's clean (noiseless) observations of the true post-edit trajectory — **not a model output** |
| **Unsteered** | model rollout from the un-edited warm-up state `h0`; the no-edit baseline |
| **True-state swap** | model rollout from the **teacher-forced post-edit state** (the model has *seen* the teleport frame). |

**Editors:**

| editor | mechanism | needs | oracle? |
|---|---|---|---|
| **Readout injection** | linear-probe pseudoinverse (`inject_state`): set the probe readout to the target positions, preserve the probe null-space; no manifold projection | linear position probe | no |
| **MLP-probe gradient** | `pim.editors.gradient_steer`: Adam on `h` minimising ‖MLP-probe(h) − target (pos,vel)‖² through a **frozen MLP (pos,vel) probe** | MLP (pos,vel) probe | no |
| **Global-PCA projection** | POCS (`manifold_steer`): alternate readout injection with projection onto the **global**-PCA subspace (90% var), 50 iterations | linear probe + global PCA | no |
| **PCA geodesic** | **iterative** constant-step walk: step toward the injection target, re-project onto a **fresh local-PCA tangent** (k=64) each iteration; K=120 (budget extension to K=600 below) | linear probe + state bank | no |
| **Decoder gradient** | Adam on `h` minimising ‖decode(h) − GT observation at the edit frame‖² through the model's decoder | **the GT observation** of the post-edit frame | **yes — oracle** |

*Naming footnote:* earlier notebooks conflated "local-tangent projection" (a **one-shot** projection onto
a single local tangent) with the **PCA geodesic** (an **iterative** walk that refits the tangent every
step). Only the iterative walk appears here.

### §4 metric definitions

| metric | formula | units | better | reference value |
|---|---|---|---|---|
| readout RMSE | `√mean((A·h_edit + b − target_pos)²)` — probe readout of the edited state vs the post-edit target positions, in state space, **at the edit step before any rollout** | position (world units) | ↓ | un-edited state ≈ 1.84 (both models, printed in [13]); 0 = readout exactly hit |
| GT next-step RMSE | `RMSE(generated obs at rollout step 1, sim clean obs at frame ef+1)` | obs intensity (0–1) | ↓ | true-state swap 0.203 (GRU) / 0.271 (RSSM) = model ceiling; unsteered 0.279 = no-edit level |
| per-step GT-trajectory RMSE | `RMSE(generated obs at step s, sim clean obs at frame ef+s)` — distance to the **time-evolving** true post-edit trajectory | obs intensity | ↓ | same references per step (Fig 4 b/e; tables in [18]) |
| step-0 →static-target RMSE | `RMSE(generated obs at step 0, static render of the edit-frame target positions)` — **step-0 direct-edit check only**: objects keep moving, so later steps must not be compared to this static render | obs intensity | ↓ | true-state swap 0.189 (GRU) / 0.259 (RSSM); unsteered 0.278 / 0.281 |
| obs-change (% of swap) | `100 · RMSE(obs_edit, obs_unsteered) / RMSE(obs_swap, obs_unsteered)` at step 0 — how much the edit moved the observation, as % of what swapping in the true post-edit state moves (the proper 100%) | % | → 100 | swap obs-change = 0.129 (GRU) / 0.059 (RSSM); 0 = nothing moved |
| ghost-ray ratio | **ghost rays** := rays where the edited object was pre-edit AND must not be post-edit (`pre_render_id==edit_obj & tgt_render_id!=edit_obj`; "ghost rays available: N" counts them over all 64 samples — here N=911). Ratio = mean step-0 intensity on ghost rays, editor ÷ unsteered | ratio | ↓ | 1 = ghost fully remains (unsteered = 1 by construction) |
| global-PCA hull residual | raw `‖h − proj_global(h)‖` onto the global 90%-var PCA subspace | ‖h‖ units | ↓ | real states ≈ 1.75 (GRU) / 2.84 (RSSM), printed in [14] |
| leave-out local-PCA residual | `‖q − proj_local(q)‖ / ‖q − local mean‖`, local PCA on the k=64 nearest bank states **excluding the query's own nearest neighbour** — without the exclusion a state already in the bank projects onto itself and the metric is a tautology | fraction | ↓ | real states ≈ 0.58 (GRU) / 0.63 (RSSM), printed in [14] and beside every use |

> *Footnotes.* (i) Older notebooks divided obs-change by the **readout-injection** obs-change (a weak
> pseudoinverse denominator); the [17] tables print that variant once for continuity, but all reported
> percentages use the true-state-swap denominator. (ii) All probes are fit in-sample: comparisons are
> load-bearing, absolute values optimistic. (iii) Decode conventions differ by one frame across
> architectures (the GRU decoder predicts the *next* observation, the RSSM decoder reconstructs the
> *current* one), so a ±1-frame offset against the sim trajectory is possible for the reference rollouts;
> it is small relative to the teleport distances being measured. (iv) The geodesic's very low leave-out
> local-PCA residual is partly by construction — its final operation is itself a projection onto a
> local-PCA plane fit from bank neighbours, so this one metric flatters it.

> **Current results (updated 2026-07-15).** Full tables in [17]–[18]; waterfalls Fig 5a/5b; scans Fig 6a/6b.
> - **The model's own belief updates slowly.** Even the **true-state swap** (which saw the teleport frame)
>   changes the observation by only RMS 0.129 (GRU) / 0.059 (RSSM) at step 0, with ghost-ray ratio 0.665 /
>   0.884 — one post-edit frame relocates the object only partially. All %-of-swap numbers are relative to
>   this sluggish (but correct) 100%.
> - **Readout injection is decoder-inert.** Readout RMSE 0.000 on both models, yet obs-change is 15.7% of
>   the swap (GRU) and **0.1%** (RSSM); ghost-ray ratio 0.985 / 1.000. On the RSSM the rollout is
>   pixel-identical to unsteered. Readable ≠ controllable in its purest form.
> - **MLP-probe gradient** does not transfer either: obs-change 43.5% / 61.5% of swap, but the ghost stays
>   (0.996 / 0.960) and GT next-step RMSE ≈ unsteered (0.276 vs 0.279 GRU; 0.273 vs 0.279 RSSM). On the RSSM
>   it drives the linear readout to 16.0 — the probe optimum it finds lies off-distribution (local-PCA resid
>   0.80 vs real 0.63).
> - **Global-PCA projection** moves the obs most among non-oracle editors on the GRU (74.5% of swap) but
>   scrambled: next-step RMSE 0.267 vs unsteered 0.279 (true-state swap: 0.203) — diffuse mass near the
>   target zone, ghost largely intact (0.928).
> - **PCA geodesic** improves the readout substantially on the GRU (1.84 → 1.24 @ K=120; 1.03 at the K=600
>   plateau — a *better* readout than the true-state swap's 1.61) while its observation barely approaches GT
>   (next-step 0.277) — readout accuracy and observation accuracy are nearly decoupled. On the RSSM it
>   barely moves (1.84 → 1.81): the injection distance, hence the constant step (0.011), is tiny.
> - **Decoder gradient (oracle)** nails the step-0 observation (0.011 / 0.039 vs the static target render)
>   and removes the ghost (ratio 0.087 / 0.088) — but the edited state is far off-manifold (leave-out
>   local-PCA resid 0.99 both; global hull resid 15.7 / 20.2 vs real 1.75 / 2.84; RSSM linear readout 210).
>   Its GRU rollout **collapses off-distribution**: GT-trajectory RMSE is back above unsteered by ~step 4
>   while its distance to the unsteered rollout stays ≈ 0.31 throughout — it never reverts to the no-edit
>   trajectory; the Fig 5a waterfall shows fragmented, speckled, non-scene-like output. The RSSM rollout
>   degrades more slowly (next-step 0.131, the best of any state here) but the target streak smears and the
>   ghost re-emerges by ~step 12, again without re-matching the unsteered rollout.
> - **No non-oracle editor beats the true-state swap on GT next-step RMSE** on either model; the only state
>   that does is the oracle decoder gradient on the RSSM (0.131 vs 0.271), and it pays with a collapsed
>   longer rollout.


In [ ]:
# [12] §4 — shared edit-set setup (model-independent): targets, sim renders, ghost/target rays, GT trajectory.
from tqdm.auto import tqdm
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene

SUBSPACE_VAR, LOCAL_VAR, LOCAL_BANK_SIZE = 0.90, 0.90, 50_000
LOCAL_K_GEO = 64                  # local-PCA neighbourhood size (diagnostic_corrections tested {16,32,64}; 64 reached furthest)
N_EDIT, N_ROLLOUT = 64, 15
K_GEO_ITERS = 120                 # geodesic budget in the head-to-head (budget extension in [24])
N_CTX = 6                         # pre-edit context frames shown in every waterfall column (sim clean obs)

N  = min(N_EDIT, edits.n_samples)
ef = edits.edit_frame

# targets: post-edit positions and (pos,vel) at the edit frame
tgt_pos_flat = edits.positions[:N, ef, :N_OBJ, :].reshape(N, N_OBJ * 2).astype(np.float32)
vel_edits    = h5py.File(edits.h5_path, "r")["velocities"][:N, ef, :N_OBJ, :].astype(np.float32)
tgt    = torch.from_numpy(tgt_pos_flat).float().to(DEVICE)                                  # (N,4) positions
tgt_pv = torch.from_numpy(np.concatenate([tgt_pos_flat, vel_edits.reshape(N, N_OBJ * 2)], 1)).float().to(DEVICE)  # (N,8) (pos,vel)

# GT reference = the SIMULATION's clean observations (never a model output)
gt_traj_obs = edits.clean_obs[:N, ef:ef + N_ROLLOUT, :].astype(np.float32)   # (N,15,R) true post-edit trajectory
ctx_obs     = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)       # (N,6,R) shared pre-edit context
gt_obs_edit_frame = torch.from_numpy(edits.clean_obs[:N, ef, :]).float().to(DEVICE)  # decoder-gradient oracle target
OBS_RES = gt_traj_obs.shape[-1]

# static renders at the edit frame: step-0 direct-edit comparison + waterfall/scan centroids and zones
sim = test.config["dataset"]["sim"]
cfg1 = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                 n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=sim["dt"], obs_res=sim["obs_res"],
                 refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                 obs_noise_std=0.0, boundary="open", always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1, 1, 1]], np.float32), (N_OBJ, 1))
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_render_id  = np.zeros((N, OBS_RES), np.int64); tgt_render_int = np.zeros((N, OBS_RES), np.float32)
pre_render_id  = np.zeros((N, OBS_RES), np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc); tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    scp = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, ridp, _ = render_scene(scp); pre_render_id[i] = ridp[0]

edit_obj = edits.edit_object[:N]
ghost_mask  = np.zeros((N, OBS_RES), bool)   # rays where the edited object was pre-edit AND must not be post-edit
target_mask = np.zeros((N, OBS_RES), bool)   # rays where the edited object must appear post-edit
for i in range(N):
    ghost_mask[i]  = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])
    target_mask[i] = (tgt_render_id[i] == edit_obj[i])

# representative samples for waterfalls/scans: largest teleport with a resolvable ghost zone
def centroid(mask_row):
    idx = np.where(mask_row)[0]; return idx.mean() if idx.size else np.nan
teleport  = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES   = list(np.argsort(teleport * has_ghost)[::-1][:3])

print(f"N={N}  edit_frame={ef}  rollout={N_ROLLOUT}  ghost rays available: {int(ghost_mask.sum())} "
      f"(over {N} samples x {OBS_RES} rays)")
print(f"waterfall/scan samples {SAMPLES} (teleport {[round(float(teleport[s]), 2) for s in SAMPLES]})")

In [ ]:
# [13] §4 — per-model prep: warm-up h0, linear position probe, frozen MLP (pos,vel) probe, global-PCA
#            subspace, local bank, and the teacher-forced post-edit state (the "True-state swap" reference).
from pim.editors import gradient_steer

@torch.no_grad()
def tf_hidden_at(model, H, obs_seqs, frame):
    """Teacher-force each sequence through `frame` (inclusive — the model SEES the teleport frame);
    return flat states (N,H)."""
    out = np.zeros((obs_seqs.shape[0], H), np.float32)
    for i in range(obs_seqs.shape[0]):
        ot = torch.from_numpy(obs_seqs[i]).float().to(DEVICE); state = None
        for t in range(frame + 1):
            _, state = model.step(ot[t].unsqueeze(0), state)
        out[i] = model.flat_state(state).squeeze(0).cpu().numpy()
    return out

def prep_model(model, states, H, name):
    # linear position probe (readout metric + Readout injection / Global-PCA projection / PCA geodesic)
    sdef = StateDefinition(name="positions", state_shape=(N_OBJ, 2), extract_fn=lambda b: b["positions"])
    lin = LinearExtractor(H, sdef, use_lstsq=True)
    lin.fit(states, pos_tf, mask=vis_tf, device=DEVICE)
    lin = lin.to(DEVICE).eval()
    A, b_, A_pinv = probe_decomposition(lin)
    # frozen MLP (pos,vel) probe (for the MLP-probe gradient editor)
    sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ * 4,), extract_fn=lambda b: b)
    mlp_pv = MLPExtractor(H, sdef_pv, mlp_hidden=128, n_epochs=30, lr=5e-3)
    pv_loss = mlp_pv.fit(states, posvel_tf, mask=vis_tf, device=DEVICE)
    mlp_pv = mlp_pv.to(DEVICE).eval()
    with torch.no_grad():
        Xs = torch.from_numpy(states[vis_tf]).float().to(DEVICE)
        pred_pv = mlp_pv(Xs).reshape(Xs.shape[0], -1).cpu().numpy()
    Ypv = posvel_tf[vis_tf]
    r2_pv = float(1 - ((pred_pv - Ypv) ** 2).sum() / ((Ypv - Ypv.mean(0)) ** 2).sum())
    # global-PCA subspace + on-device local bank
    sub = fit_state_subspace(states, var_threshold=SUBSPACE_VAR)
    sub = replace(sub, mean=sub.mean.to(DEVICE), basis=sub.basis.to(DEVICE),
                  explained_variance_ratio=sub.explained_variance_ratio.to(DEVICE))
    bank_all = states.reshape(-1, H)
    bank_idx = np.random.RandomState(0).choice(bank_all.shape[0], size=min(LOCAL_BANK_SIZE, bank_all.shape[0]), replace=False)
    bank = torch.from_numpy(bank_all[bank_idx]).float().to(DEVICE)
    # warm-up on PRE-edit observations + teacher-forced POST-edit state
    warm = eval.warm_up_to_edit(model, edits.obs[:N], ef, n_viz=N, n_ctx_show=8, device=DEVICE)
    h0 = torch.from_numpy(warm.h_at_edit[:N]).float().to(DEVICE)
    h_swap = torch.from_numpy(tf_hidden_at(model, H, edits.obs[:N], ef)).float().to(DEVICE)
    print(f"{name}: MLP (pos,vel) probe — final train loss {pv_loss:.5f}, in-sample R2 {r2_pv:.3f}")
    return dict(model=model, H=H, A=A, b=b_, A_pinv=A_pinv, mlp_pv=mlp_pv, sub=sub, bank=bank, h0=h0, h_swap=h_swap)

WM = {"GRU": prep_model(gru, states_gru, H_GRU, "GRU"),
      "RSSM": prep_model(rssm, states_rssm, H_RSSM, "RSSM")}

def readout(P, h): return h @ P["A"].T + P["b"]
def readout_rmse(P, h): return float((readout(P, h) - tgt).pow(2).mean().sqrt())
for m, P in WM.items():
    print(f"{m}: un-edited readout RMSE {readout_rmse(P, P['h0']):.4f} | true-state-swap readout RMSE {readout_rmse(P, P['h_swap']):.4f}")

In [ ]:
# [14] §4 — leave-out local-PCA residual (fraction) + per-model real-state references.
#      Local PCA tangent on the k=64 nearest bank states, EXCLUDING the query's own nearest neighbour:
#      without the exclusion, a state that is in (or extremely near) the bank projects onto itself and the
#      residual is tautologically ~0. Reported as ||q - proj|| / ||q - local mean||.
@torch.no_grad()
def loo_local_resid(h_batch, bank, k_neighbors=LOCAL_K_GEO, leave_out=True, n_probe=100, var_threshold=LOCAL_VAR):
    hb = h_batch if isinstance(h_batch, torch.Tensor) else torch.as_tensor(h_batch, device=DEVICE, dtype=torch.float32)
    fracs = []
    for i in range(min(n_probe, hb.shape[0])):
        q = hb[i].reshape(-1)
        d = torch.cdist(q[None], bank)[0]
        kk = k_neighbors + (1 if leave_out else 0)
        idx = torch.topk(d, min(kk, bank.shape[0]), largest=False).indices
        if leave_out: idx = idx[1:]
        sub = _pca_subspace(bank[idx], n_components=None, var_threshold=var_threshold)
        proj = project_to_subspace(q[None], sub)[0]
        fracs.append(float((q - proj).norm()) / max(float((q - sub.mean).norm()), 1e-9))
    return float(np.mean(fracs))

REAL_LOO, REAL_GLOB = {}, {}
for m, P in WM.items():
    REAL_LOO[m]  = loo_local_resid(P["bank"][:200], P["bank"], n_probe=200)
    REAL_GLOB[m] = float(offmanifold_residual(P["bank"][:2000], P["sub"]).mean())
    print(f"{m}: real-state references — leave-out local-PCA residual {REAL_LOO[m]:.3f} (fraction), "
          f"global-PCA hull residual {REAL_GLOB[m]:.3f} (||h|| units)")

In [ ]:
# [15] §4 — run the five editors on both world models.
ED_ORDER = ["Readout injection", "MLP-probe gradient", "Global-PCA projection", "PCA geodesic", "Decoder gradient"]

@torch.no_grad()
def pca_geodesic(P, h_start, target, k_local=LOCAL_K_GEO, const_step=None, k_iters=K_GEO_ITERS,
                 plateau_window=None, plateau_tol=0.01, desc="PCA geodesic"):
    """Iterative constant-step walk: step toward the injection target, re-project onto a fresh local-PCA
    tangent each iteration. Optional plateau early-stop: stop a sample when readout RMSE improves by
    < plateau_tol (relative) over the last `plateau_window` iterations. Log rows are NaN after a stop."""
    A, b_, A_pinv, bank = P["A"], P["b"], P["A_pinv"], P["bank"]
    if const_step is None:
        const_step = 0.34 * float((inject_state(h_start, target, A, A_pinv, b_) - h_start).norm(dim=-1).mean())
    Nn = h_start.shape[0]
    h_out = torch.empty_like(h_start)
    rmse_log = np.full((Nn, k_iters + 1), np.nan)
    for i in tqdm(range(Nn), desc=desc, leave=False):
        h = h_start[i:i + 1]; t = target[i:i + 1]
        rmse_log[i, 0] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
        for kk in range(k_iters):
            d = inject_state(h, t, A, A_pinv, b_) - h
            nrm = d.norm(); dhat = d / nrm if float(nrm) > 1e-12 else d
            h_step = h + const_step * dhat
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_local, var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
            h = project_to_subspace(h_step, sub)
            rmse_log[i, kk + 1] = float((h @ A.T + b_ - t).pow(2).mean().sqrt())
            if plateau_window is not None and kk + 1 >= plateau_window:
                prev = rmse_log[i, kk + 1 - plateau_window]
                if (prev - rmse_log[i, kk + 1]) < plateau_tol * max(prev, 1e-9):
                    break
        h_out[i] = h[0]
    return h_out, rmse_log, const_step

def decoder_grad_edit(model, h_init, target_obs, n_iter=400, lr=0.05):
    """ORACLE editor: Adam on h minimising ||decode(h) - GT observation at the edit frame||^2."""
    h = h_init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):   # allow backward through the eval-mode recurrent model
        for _ in range(n_iter):
            pred = model.decode(model.state_from_flat(h))
            loss = ((pred - target_obs) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach(), float(loss.item())

EDITS4, GEO_LOG, CONST_STEP = {}, {}, {}
for m, P in WM.items():
    h0 = P["h0"]; A, b_, A_pinv = P["A"], P["b"], P["A_pinv"]
    E = {}
    E["Readout injection"] = inject_state(h0, tgt, A, A_pinv, b_)          # bare pseudoinverse, no projection
    outs = []
    for i in tqdm(range(N), desc=f"MLP-probe gradient ({m})", leave=False):
        h_i, _ = gradient_steer(h0[i:i + 1], tgt_pv[i:i + 1], P["mlp_pv"], n_steps=200, lr=0.01)
        outs.append(h_i)
    E["MLP-probe gradient"] = torch.cat(outs, 0)
    E["Global-PCA projection"] = manifold_steer(h0, tgt, lambda h, t: inject_state(h, t, A, A_pinv, b_), P["sub"], n_iters=50)
    E["PCA geodesic"], GEO_LOG[m], CONST_STEP[m] = pca_geodesic(P, h0, tgt, desc=f"PCA geodesic ({m})")
    E["Decoder gradient"], dec_loss = decoder_grad_edit(P["model"], h0, gt_obs_edit_frame)
    EDITS4[m] = E
    print(f"{m}: const_step={CONST_STEP[m]:.4f} | geodesic readout RMSE {np.nanmean(GEO_LOG[m][:, 0]):.3f} -> "
          f"{np.nanmean(GEO_LOG[m][:, -1]):.3f} | decoder-gradient final decode MSE {dec_loss:.6f}")
    print("   readout RMSE: " + " | ".join(f"{n} {readout_rmse(P, h):.3f}" for n, h in E.items()))

In [ ]:
# [16] §4 — model rollouts from every reference/editor state (rollout step s targets sim frame ef+s).
REF_ORDER = ["Unsteered", "True-state swap"]
COL = {"GT (sim)": "k", "Unsteered": OK["grey"], "True-state swap": "#56B4E9",
       "Readout injection": OK["yellow"], "MLP-probe gradient": OK["orange"],
       "Global-PCA projection": OK["green"], "PCA geodesic": OK["blue"], "Decoder gradient": OK["pink"]}

@torch.no_grad()
def rollout_from_flat(model, h_array, n_rollout):
    obs_all = []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, _ = _rollout(model, h, n_rollout)
        obs_all.append(o)
    return np.stack(obs_all)

ROLL = {}
for m, P in WM.items():
    states4 = {"Unsteered": P["h0"], "True-state swap": P["h_swap"], **EDITS4[m]}
    ROLL[m] = {n: rollout_from_flat(P["model"], h.detach().cpu().numpy(), N_ROLLOUT) for n, h in states4.items()}
    print(f"{m}: {len(ROLL[m])} rollout sets, each {ROLL[m]['Unsteered'].shape}")

In [ ]:
# [17] §4 — metric suite (same metrics, same units, both models); formulas in the §4 definitions table.
from IPython.display import Markdown

def rms(a, b): return float(np.sqrt(((a - b) ** 2).mean()))

def md_table(data, columns, row_hdr=""):
    """Clearly-demarcated rendered-markdown table (matches the §1/§2 table style; no pandas dependency).
    data: {row_name: {key: value}}; columns: list of (key, display_name, format_str)."""
    lines = ["| " + row_hdr + " | " + " | ".join(d for _, d, _ in columns) + " |",
             "|" + "---|" * (len(columns) + 1)]
    for rn, vals in data.items():
        cells = []
        for k, _, f in columns:
            v = vals[k]
            cells.append("nan" if isinstance(v, float) and np.isnan(v) else f.format(v))
        lines.append("| **" + str(rn) + "** | " + " | ".join(cells) + " |")
    return Markdown("\n".join(lines))

METRICS, STEP_RMSE, SWAP_CHG = {}, {}, {}
for m, P in WM.items():
    obs_u = ROLL[m]["Unsteered"]
    swap_chg = rms(ROLL[m]["True-state swap"][:, 0, :], obs_u[:, 0, :])   # the proper 100% obs-change reference
    SWAP_CHG[m] = swap_chg
    rows = {}
    for n in REF_ORDER + ED_ORDER:
        o = ROLL[m][n]
        h = {"Unsteered": P["h0"], "True-state swap": P["h_swap"]}.get(n)
        if h is None:
            h = EDITS4[m][n]
        chg = rms(o[:, 0, :], obs_u[:, 0, :])
        if ghost_mask.sum() > 0:
            ghost = float(o[:, 0, :][ghost_mask].mean() / max(obs_u[:, 0, :][ghost_mask].mean(), 1e-6))
        else:
            ghost = float("nan")
        rows[n] = dict(
            readout    = readout_rmse(P, h),
            nextstep   = rms(o[:, 1, :], gt_traj_obs[:, 1, :]),
            to_tgt0    = rms(o[:, 0, :], tgt_render_int),
            obs_chg    = chg,
            pct_swap   = 100 * chg / max(swap_chg, 1e-9),
            ghost      = ghost,
            loo_resid  = loo_local_resid(h, P["bank"], n_probe=min(64, N)),
            glob_resid = float(offmanifold_residual(h, P["sub"]).mean()),
        )
    METRICS[m] = rows
    STEP_RMSE[m] = {n: [rms(ROLL[m][n][:, s, :], gt_traj_obs[:, s, :]) for s in range(N_ROLLOUT)]
                    for n in REF_ORDER + ED_ORDER}

M4_COLS = [("readout", "readout RMSE (pos)", "{:.3f}"), ("nextstep", "GT next-step RMSE (obs)", "{:.3f}"),
           ("to_tgt0", "step-0 vs static target (obs)", "{:.3f}"), ("obs_chg", "obs-change (obs)", "{:.3f}"),
           ("pct_swap", "% of swap", "{:.1f}"), ("ghost", "ghost-ray ratio", "{:.3f}"),
           ("loo_resid", "leave-out local-PCA resid (frac)", "{:.3f}"), ("glob_resid", "global-PCA hull resid (‖h‖)", "{:.3f}")]
for m in WM:
    pinv_pct = 100 * METRICS[m]["Decoder gradient"]["obs_chg"] / max(METRICS[m]["Readout injection"]["obs_chg"], 1e-9)
    print(f"=== {m} — editor metrics (references first) ===")
    print(f"    references: real-state leave-out local-PCA resid {REAL_LOO[m]:.3f} | real-state global hull resid "
          f"{REAL_GLOB[m]:.3f} | true-state-swap obs-change {SWAP_CHG[m]:.4f} (the 100% denominator)")
    print(f"    (footnote variant: decoder-gradient obs-change / readout-injection obs-change = {pinv_pct:.0f}% — "
          f"the old weak pseudoinverse denominator, not used elsewhere)")
    display(md_table(METRICS[m], M4_COLS, row_hdr=f"{m} editor"))

metrics4 = METRICS["GRU"]   # per-model alias used by §5

In [ ]:
# [18] §4 — persistence, two complementary per-step tables (both models):
#      (i)  RMSE(generated obs at step s, sim clean obs at frame ef+s) — does the edit track the true
#           post-edit trajectory over the rollout?
#      (ii) RMSE(generated obs at step s, unsteered rollout at step s) — distinguishes an edit that
#           DISSOLVES back into the no-edit rollout (curve -> ~0: "reverts") from one whose output leaves
#           both trajectories (stays large while (i) is also large: "collapses off-distribution").
#      What the failing rollout actually looks like is read from the Fig 5a/5b waterfalls.
steps = np.arange(N_ROLLOUT)
DIST_TO_UNSTEERED = {}
for m in WM:
    DIST_TO_UNSTEERED[m] = {n: [rms(ROLL[m][n][:, s, :], ROLL[m]["Unsteered"][:, s, :]) for s in steps]
                            for n in ["True-state swap"] + ED_ORDER}
    cols_i  = [(n, n, "{:.3f}") for n in REF_ORDER + ED_ORDER]
    data_i  = {f"step {s}": {n: STEP_RMSE[m][n][s] for n in REF_ORDER + ED_ORDER} for s in steps}
    cols_ii = [(n, n, "{:.3f}") for n in ["True-state swap"] + ED_ORDER]
    data_ii = {f"step {s}": {n: DIST_TO_UNSTEERED[m][n][s] for n in ["True-state swap"] + ED_ORDER} for s in steps}
    print(f"=== {m} — (i) per-step GT-trajectory RMSE: RMSE(gen obs @ step s, sim clean obs @ frame ef+s) ===")
    display(md_table(data_i, cols_i, row_hdr="rollout step"))
    print(f"=== {m} — (ii) distance to the unsteered rollout: RMSE(gen obs @ step s, unsteered @ step s) ===")
    display(md_table(data_ii, cols_ii, row_hdr="rollout step"))

In [ ]:
# [19] Fig 4 — editing head-to-head, one ROW per world model:
#      (left) readout accuracy vs next-step observation accuracy, (mid) per-step GT-trajectory RMSE,
#      (right) leave-out local-PCA residual vs the real-state reference.
from matplotlib.lines import Line2D
fig, axes = plt.subplots(2, 3, figsize=(18, 9.2))
for r, m in enumerate(["GRU", "RSSM"]):
    Mx = METRICS[m]
    pa, pb, pc = axes[r]
    # (a/d) scatter: state-space accuracy vs observation-space accuracy (legend is figure-level, outside axes)
    for n in REF_ORDER + ED_ORDER:
        pa.scatter(Mx[n]["readout"], Mx[n]["nextstep"], s=95, color=COL[n],
                   marker="s" if n in REF_ORDER else "o", edgecolor="k", zorder=3)
    pa.set_xlabel("readout RMSE at edit step (position units)")
    pa.set_ylabel("GT next-step RMSE (obs intensity)")
    pa.set_title(f"({'ad'[r]}) {m} — readout accuracy vs next-step observation accuracy")
    style_ax(pa)
    # (b/e) per-step distance to the true post-edit trajectory
    for n in REF_ORDER + ED_ORDER:
        pb.plot(steps, STEP_RMSE[m][n], color=COL[n], lw=1.6, marker="o", ms=3)
    pb.set_xlabel("rollout step (0 = edit frame)")
    pb.set_ylabel("RMSE(generated obs, true post-edit obs)")
    pb.set_title(f"({'be'[r]}) {m} — per-step GT-trajectory RMSE")
    style_ax(pb)
    # (c/f) manifold residency
    names_c = REF_ORDER + ED_ORDER
    vals = [Mx[n]["loo_resid"] for n in names_c]
    pc.bar(range(len(names_c)), vals, color=[COL[n] for n in names_c], alpha=0.9)
    for i, v in enumerate(vals):
        pc.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    pc.axhline(REAL_LOO[m], color="0.3", ls="--", lw=1.4)
    pc.text(len(names_c) - 0.55, REAL_LOO[m] + 0.015, f"real states {REAL_LOO[m]:.2f}", ha="right", fontsize=8, color="0.3")
    pc.set_xticks(range(len(names_c))); pc.set_xticklabels(names_c, rotation=25, ha="right", fontsize=8)
    pc.set_ylabel("leave-out local-PCA residual (fraction)")
    pc.set_title(f"({'cf'[r]}) {m} — manifold residency of the edited state")
    style_ax(pc)
handles = [Line2D([0], [0], marker="s" if n in REF_ORDER else "o", linestyle="none", markersize=8,
                  markerfacecolor=COL[n], markeredgecolor="k", label=n) for n in REF_ORDER + ED_ORDER]
fig.legend(handles=handles, loc="upper center", ncol=7, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0.965))
fig.suptitle("Fig 4 — Editing head-to-head: state-space accuracy, observation-space accuracy, manifold residency (GRU top, RSSM bottom)",
             y=0.995, fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig(f"{OUT}/fig4_editor_metrics.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

### §4a — GRU

Waterfalls (Fig 5a) and step-0 observation scans (Fig 6a) for the GRU. In every waterfall column the
first 6 rows are the simulation's clean pre-edit observations (shared context); below the orange dashed
edit-frame line, the "GT (sim)" column continues with the simulation's true post-edit observations while
every other column shows the model's own rollout from its (edited) hidden state.

> **Current results (updated 2026-07-15).** In Fig 5a/6a: **Readout injection** is visually
> indistinguishable from Unsteered (obs-change 15.7% of swap, ghost-ray ratio 0.985). **MLP-probe
> gradient** adds diffuse changes without emptying the ghost (0.996). **Global-PCA projection** deposits
> diffuse mass around the target zone but leaves the ghost streak (0.928). **PCA geodesic** shifts/darkens
> the scene slightly (ghost 0.914) despite reaching the best non-oracle readout (1.24). **Decoder
> gradient** reproduces the target scan almost exactly at step 0 (0.011 vs the static render, ghost 0.087),
> then the rollout **collapses off-distribution**: fragmented, speckled output (rightmost Fig 5a column),
> GT-trajectory RMSE back above unsteered by ~step 4, distance to the unsteered rollout flat at ≈ 0.31 —
> a collapse, not a revert. The **True-state swap** column shows the model's belief lag: partial
> appearance at the target with ghost-ray ratio 0.665 and next-step RMSE 0.203 (vs unsteered 0.279).


In [ ]:
# [20] Fig 5a — GRU editor waterfalls (dark, world_model_eval styling: gray cmap, dashed edit-frame line).
DARK_BG, DARK_TEXT, DARK_TICK, EDIT_LINE = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
WATERFALL_COLS = ["GT (sim)"] + REF_ORDER + ED_ORDER   # 8 full-size columns

def editor_waterfall_fig(m, fig_tag, fname):
    n_cols = len(WATERFALL_COLS)
    fig, axes = plt.subplots(len(SAMPLES), n_cols, figsize=(3.2 * n_cols, 3.6 * len(SAMPLES)),
                             squeeze=False, facecolor=DARK_BG)
    for r, smp in enumerate(SAMPLES):
        tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp])
        pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
        for c, n in enumerate(WATERFALL_COLS):
            ax = axes[r][c]
            post = gt_traj_obs[smp] if n == "GT (sim)" else ROLL[m][n][smp]
            panel = np.clip(np.concatenate([(edits.clean_obs[smp, ef - N_CTX:ef, :].astype(np.float32) if n == "GT (sim)" else ctx_obs[smp]), post], axis=0), 0, 1)
            ax.set_facecolor(DARK_BG)
            for spine in ax.spines.values():
                spine.set_edgecolor(DARK_TICK)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            ax.axhline(N_CTX - 0.5, color=EDIT_LINE, lw=1.2, ls="--", alpha=0.8)
            if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(n, fontsize=10, color=DARK_TEXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.2f})\nsim frame", fontsize=8, color=DARK_TEXT)
                ax.set_yticks([0, N_CTX, N_CTX + 5, N_CTX + 10])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 5, ef + 10])
            else:
                ax.set_yticks([])
            ax.set_xlabel("ray position", fontsize=8, color=DARK_TEXT)
            ax.tick_params(colors=DARK_TICK, labelsize=7)
    handles = [Line2D([0], [0], color="#00E676", lw=2.2, label="target location (post-edit)"),
               Line2D([0], [0], color="#FF5252", ls="--", lw=2.2, label="ghost location (pre-edit)"),
               Line2D([0], [0], color=EDIT_LINE, ls="--", lw=2.2, label=f"edit frame ({N_CTX} rows above = observed context (noisy for editors; clean under GT))")]
    fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=10, frameon=False,
               labelcolor=DARK_TEXT, bbox_to_anchor=(0.5, 0.975))
    fig.suptitle(f"Fig {fig_tag} — {m} editor waterfalls: {N_CTX} observed context frames (noisy for editors; clean under GT), then model rollout from the edited state",
                 y=0.998, fontsize=13, color=DARK_TEXT)
    fig.tight_layout(rect=[0, 0, 1, 0.945])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK_BG)
    display(fig); plt.close(fig)

editor_waterfall_fig("GRU", "5a", "fig5a_waterfalls_gru.png")

In [ ]:
# [21] Fig 6a — GRU: generated observation at rollout step 0 (direct edit), every editor overlaid
#      (geodesic_walk_k150 scan style; static target render is the sanctioned step-0 comparison).
SCAN_ORDER = REF_ORDER + ED_ORDER

def scan_fig(m, fig_tag, fname):
    rays = np.arange(OBS_RES)
    fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(11.5, 3.1 * len(SAMPLES)), squeeze=False)
    for r, smp in enumerate(SAMPLES):
        ax = axes[r][0]
        ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="target render (static, edit frame)", zorder=6)
        gz = np.where(ghost_mask[smp])[0]
        if gz.size:
            ax.axvspan(gz.min() - 0.5, gz.max() + 0.5, color="red", alpha=0.10, zorder=0,
                       label="ghost zone (edited object pre-edit only)")
        tz = np.where(target_mask[smp])[0]
        if tz.size:
            ax.axvspan(tz.min() - 0.5, tz.max() + 0.5, color="green", alpha=0.10, zorder=0,
                       label="target zone (edited object post-edit)")
        for n in SCAN_ORDER:
            ax.plot(rays, ROLL[m][n][smp, 0], color=COL[n], lw=1.5, alpha=0.9, label=n, zorder=3)
        ax.set_title(f"sample {smp} (edited object {edit_obj[smp]}, teleport {teleport[smp]:.2f})", fontsize=10)
        ax.set_xlabel("ray index"); ax.set_ylabel("intensity"); ax.set_ylim(-0.02, 1.05)
        style_ax(ax)
        if r == 0:
            ax.legend(fontsize=7, ncol=3, loc="upper right")
    fig.suptitle(f"Fig {fig_tag} — {m}: generated observation at rollout step 0 (direct edit), per editor",
                 y=1.0, fontsize=12)
    fig.tight_layout()
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight")
    display(fig); plt.close(fig)

scan_fig("GRU", "6a", "fig6a_scans_gru.png")

### §4b — RSSM

The same editor line-up, waterfalls (Fig 5b) and step-0 observation scans (Fig 6b) for the refined RSSM
(flat state = cat[det 256, stoch 64]). Layout identical to §4a: 6 rows of sim clean-obs context, then the
model rollout below the edit-frame line; the "GT (sim)" column is the simulation itself.

> **Current results (updated 2026-07-15).** In Fig 5b/6b: **Readout injection** changes the observation by
> **0.1% of a swap** (ghost-ray ratio 1.000) — its column is pixel-identical to Unsteered; the RSSM
> position-probe direction is fully decoder-inert, sharper than the GRU's 15.7%. **MLP-probe gradient**
> and **Global-PCA projection** move the obs (61.5% / 61.4% of swap) but scrambled — GT next-step RMSE
> 0.273 / 0.283 vs unsteered 0.279. **PCA geodesic** barely moves anything (readout 1.84 → 1.81; the
> injection distance, hence its constant step 0.011, is tiny). **Decoder gradient** hits the step-0
> observation (0.039) and uniquely achieves next-step RMSE 0.131 (better than the true-state swap's
> 0.271) — but drives the state absurdly far off-manifold (linear readout 210, global hull resid 20.2 vs
> real 2.84, leave-out local-PCA resid 0.99), and over the rollout the target streak smears while the
> ghost streak re-emerges (≈ unsteered GT-trajectory RMSE by step 14) without ever re-matching the
> unsteered rollout (distance stays ≈ 0.26–0.31). The RSSM **True-state swap** is even more sluggish than
> the GRU's: obs-change 0.059, ghost-ray ratio 0.884.


In [ ]:
# [22] Fig 5b — RSSM editor waterfalls (same layout/styling as Fig 5a).
editor_waterfall_fig("RSSM", "5b", "fig5b_waterfalls_rssm.png")

In [ ]:
# [23] Fig 6b — RSSM: generated observation at rollout step 0 (direct edit), per editor (same layout as Fig 6a).
scan_fig("RSSM", "6b", "fig6b_scans_rssm.png")

### §4 — PCA-geodesic budget extension: was K=120 simply too few iterations?

The head-to-head above runs the geodesic for K=120 iterations. Here the same constant-step walk (k=64,
same step size) gets **K=600** on 32 samples with a **plateau early-stop** (a sample stops once its
readout RMSE improves by less than 1% over 50 iterations). The question: does the readout gap keep
closing with more budget, or does the walk asymptote short of the target readout?

> **Current results (updated 2026-07-15).** **The walk asymptotes short of the target readout — it did
> not just need longer.** GRU: mean readout RMSE 1.75 → 1.08 at K=120 → **1.03 at the plateau**; all 32
> samples triggered the early-stop (median stop iteration 135) and the mean curve is flat from ~iteration
> 200 onward (Fig 6c-a). The plateau sits below the true-state-swap readout (1.61) but far above 0 — and
> the corresponding observations still barely move toward GT (§4 tables). RSSM: **no descent at all** —
> 1.80 → 1.75, median stop at iteration 51 (the earliest the stop can fire), flat to K=600 (Fig 6c-b).


In [ ]:
# [24] §4 — PCA-geodesic budget extension: K=600 iterations, 32 samples, plateau early-stop; Fig 6c.
K_EXT, N_EXT, PLATEAU_W, PLATEAU_TOL = 600, 32, 50, 0.01
GEO_EXT = {}
for m, P in WM.items():
    _, log_ext, _ = pca_geodesic(P, P["h0"][:N_EXT], tgt[:N_EXT], k_iters=K_EXT,
                                 plateau_window=PLATEAU_W, plateau_tol=PLATEAU_TOL,
                                 const_step=CONST_STEP[m], desc=f"geodesic K={K_EXT} ({m})")
    GEO_EXT[m] = log_ext

def ffill_rows(log):
    """Forward-fill NaNs (post-early-stop) with the last valid value: a stopped sample keeps its plateau RMSE."""
    out = log.copy()
    for i in range(out.shape[0]):
        last = out[i, 0]
        for j in range(out.shape[1]):
            if np.isnan(out[i, j]): out[i, j] = last
            else: last = out[i, j]
    return out

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))
summary_rows = {}
for k, (ax, m) in enumerate(zip(axes, ["GRU", "RSSM"])):
    log = GEO_EXT[m]; filled = ffill_rows(log)
    stop_iter = np.array([int(np.max(np.where(~np.isnan(log[i]))[0])) for i in range(log.shape[0])])
    for i in range(log.shape[0]):
        ax.plot(filled[i], color=COL["PCA geodesic"], alpha=0.12, lw=0.8)
    ax.plot(filled.mean(0), color=COL["PCA geodesic"], lw=2.4, label="mean readout RMSE")
    ax.axhline(METRICS[m]["True-state swap"]["readout"], color=COL["True-state swap"], ls="--", lw=1.4,
               label=f"true-state-swap readout RMSE ({METRICS[m]['True-state swap']['readout']:.3f})")
    ax.axvline(K_GEO_ITERS, color="0.4", ls=":", lw=1.2, label=f"head-to-head budget K={K_GEO_ITERS}")
    ax.set_xlabel("geodesic iteration"); ax.set_ylabel("readout RMSE (position units)")
    ax.set_title(f"({'ab'[k]}) {m} — readout RMSE vs iteration (K={K_EXT}, {N_EXT} samples)")
    ax.legend(fontsize=8); style_ax(ax)
    summary_rows[m] = dict(rmse_at_iter0=float(filled[:, 0].mean()),
                           rmse_at_iter120=float(filled[:, K_GEO_ITERS].mean()),
                           rmse_final=float(filled[:, -1].mean()),
                           median_stop_iter=int(np.median(stop_iter)),
                           frac_stopped_early=float((stop_iter < K_EXT).mean()))
fig.suptitle("Fig 6c — PCA-geodesic budget extension: readout RMSE vs iteration (plateau early-stop: <1% improvement over 50 iters)",
             y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/fig6c_geodesic_budget.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
EXT_COLS = [("rmse_at_iter0", "readout RMSE @ iter 0", "{:.4f}"), ("rmse_at_iter120", "@ iter 120", "{:.4f}"),
            ("rmse_final", "@ final", "{:.4f}"), ("median_stop_iter", "median stop iter", "{:d}"),
            ("frac_stopped_early", "frac stopped early", "{:.2f}")]
display(md_table(summary_rows, EXT_COLS, row_hdr="model"))

---
## §5 — Summary — what these experiments say about the learned state

This is the one section where we interpret. First the measured quantities (dated, like every result in
this notebook), then our reading, clearly marked as interpretation. Fig 7 gives the two summary panels;
cell [26] collects every headline number in demarcated tables.

> **Current results (updated 2026-07-15) — the quantities.**
> - **Geometry (§1).** Model-free intrinsic dimension is close to, and slightly below, the physical 8 for
>   the GRU (TwoNN 5.2, MLE 6.9); the RSSM sits above it (TwoNN ≈ 9.6, MLE ≈ 10.0). Both sit far below
>   their linear hulls (38 / 35 PCA components at 90% variance), and the local tangent rotates ≈ 56°
>   (GRU) / ≈ 65° (RSSM) between neighbouring states — a low-dimensional but strongly curved embedding.
> - **Recoverability (§2).** Position reads out at linear R² ≈ 0.84–0.85 and MLP R² ≈ 0.96–0.97 (both
>   models). Velocity reads out of a *single* hidden state at MLP R² ≈ 0.93–0.94 (late-t), and a two-frame
>   window adds at most 0.007 — a nonlinear but instantaneous readout. RSSM position lives in the
>   deterministic core (det-only linear R² 0.84 ≈ full 0.85; stochastic-`s`-only 0.58).
> - **Canonicality (§3).** ≈ 34% of ‖h‖ is not explained by any function g(pos,vel) we fit (GRU MLP fiber
>   residual 0.337; R² of g on `h` ≈ 0.86) — `h` is **largely but not fully** a function of the physical
>   state. The RSSM deterministic core is at the same level (0.368); its stochastic `s` is mostly not a
>   function of (pos,vel) (residual 0.891), as expected for a KL-regularised latent.
> - **Editing (§4) — reference scale first.** Even the **true-state swap** — the model that actually saw
>   the teleport frame — moves the step-0 observation by only 0.129 (GRU) / 0.059 (RSSM), with ghost-ray
>   ratio 0.665 / 0.884: single-frame belief updates are inherently sluggish, and every editor is graded
>   against this correct-but-partial 100%.
> - **Editing (§4) — non-oracle editors.** **Readout injection**: readout RMSE 0.000 on both models, yet
>   obs-change only 15.7% (GRU) / 0.1% (RSSM) of the swap, ghost-ray ratio 0.985 / 1.000. **MLP-probe
>   gradient**: moves the observation (43.5% / 61.5% of swap) but not toward the true trajectory — GT
>   next-step RMSE 0.276 / 0.273 vs unsteered 0.279, ghost 0.996 / 0.960. **Global-PCA projection**: the
>   largest non-oracle obs-change on the GRU (74.5% of swap) at next-step 0.267, ghost 0.928. **PCA
>   geodesic**: the best non-oracle readout on the GRU (1.84 → 1.24 at K=120) with next-step 0.277; at
>   K=600 the walk **asymptotes** at ≈ 1.03 (below the true-state swap's readout 1.61, far above the
>   target 0; all 32 samples early-stop; RSSM: no descent, 1.80 → 1.75) — the remaining readout gap does
>   not close with a 5× iteration budget. No non-oracle editor beats the true-state swap on GT next-step
>   RMSE on either model.
> - **Editing (§4) — the oracle.** The **decoder gradient** hits the step-0 observation (0.011 / 0.039 vs
>   the static target render; ghost 0.087 / 0.088) but parks the state far off-manifold (leave-out
>   local-PCA residual 0.99 on both models, vs real-state 0.58 / 0.63; global hull residual 15.7 / 20.2 vs
>   real 1.75 / 2.84). Its GRU rollout then **collapses off-distribution**: fragmented, non-scene-like
>   output, GT-trajectory RMSE back above unsteered by ≈ step 4 while staying ≈ 0.31 away from the
>   unsteered rollout — degeneration, not a return to the no-edit trajectory. The RSSM rollout degrades
>   more slowly (next-step 0.131, the best of any state here) but the target streak smears and the ghost
>   re-grows by ≈ step 12.

> **Our reading (interpretation — confined to this section; updated 2026-07-15).**
> - **Largely, but not fully, a function of the physical state.** With R² of g(pos,vel) on `h` ≈ 0.86 and
>   fiber residual ≈ 0.34, `h` tracks (pos,vel) closely without reducing to it. Given §0 — under process
>   and observation noise the predictively-sufficient statistic is a *belief* over (pos,vel), legitimately
>   more than 8-dimensional — we read the residual as belief/filter content plus training scaffolding, in
>   unknown proportion. It is a graded quantity, so we avoid the binary label "non-canonical" as a verdict.
> - **State-space accuracy and observation-space accuracy are nearly decoupled, and it is not a budget
>   problem.** The probe readout can be driven to 0.000 with almost no effect on the rollout; every
>   non-oracle editor lands within 0.012 of the unsteered GT next-step RMSE regardless of its readout
>   accuracy; and the K=600 geodesic shows the gap is an asymptote, not an unfinished descent. Our
>   reading: for these models the position-probe direction is close to decoder-null, and on-manifold
>   motion toward the target readout does not traverse the directions the decoder and dynamics respond to.
> - **The oracle marks the price of leaving the manifold.** The decoder gradient proves the decoder *can*
>   be driven to the target observation from `h` — but only from a state the dynamics cannot continue
>   (leave-out residual 0.99 vs real 0.58 / 0.63), and the rollout **collapses** rather than reverting.
>   Our reading: a persistent edit needs *both* manifold residency *and* movement along decoder-visible
>   directions; no editor tested here achieves both at once.
> - **Architecture-independence.** The refined RSSM reproduces each pattern — curved low-dimensional
>   geometry, nonlinear-instantaneous velocity, det-core fiber residual 0.368 vs GRU 0.337, a
>   decoder-inert probe direction (obs-change 0.1% of swap) — with the world content in its deterministic
>   core, while the stochastic `s` holds neither position (R² 0.58) nor (pos,vel) structure (residual
>   0.891). On these measurements the KL structure buys no additional canonicality and no additional
>   controllability.
> - **The organizing hypothesis, restated as hypothesis.** These results are consistent with — not proof
>   of — the claim that editability requires a canonical, factored, predictively-sufficient state. The §4
>   reference scale adds a caveat in the other direction: part of the editing difficulty is the model's own
>   belief sluggishness (even the true-state swap relocates the object only partially in one frame), not
>   state entanglement alone.

In [ ]:
# [25] Fig 7 — Summary: (a) recoverability & canonicality scores per world model;
#      (b) editing head-to-head across world models — readout accuracy vs next-step observation accuracy.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8), gridspec_kw={"width_ratios": [1.0, 1.35]})

# (a) capability bars — the same metric per group for every model (grows by adding a bar per model);
#     canonicality uses the deterministic core for the RSSM (the §3 architecture comparison).
ax = axes[0]
groups = ["position R²\n(MLP probe)", "velocity R²\n(single-frame MLP, late-t)", "1 − fiber residual\n(MLP g; RSSM: det core)"]
bar_vals = {
    "GRU":  [pos_res[("GRU", "mlp")]["r2"],  vel_res[("GRU", "late")][("sf", "mlp")]["r2"],  1 - gru_r],
    "RSSM": [pos_res[("RSSM", "mlp")]["r2"], vel_res[("RSSM", "late")][("sf", "mlp")]["r2"], 1 - det_r],
}
mcolor = {"GRU": OK["blue"], "RSSM": OK["orange"]}
x = np.arange(len(groups)); w = 0.8 / len(bar_vals)
for k, (m, vals) in enumerate(bar_vals.items()):
    off = (k - (len(bar_vals) - 1) / 2) * w
    ax.bar(x + off, vals, w * 0.92, color=mcolor[m], label=m)
    for i, v in enumerate(vals):
        ax.text(x[i] + off, v + 0.015, f"{v:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=8.5)
ax.set_ylim(0, 1.12); ax.set_ylabel("score (0–1, higher is better)")
ax.set_title("(a) recoverability and canonicality scores per world model")
ax.legend(fontsize=8, loc="lower right"); style_ax(ax)

# (b) ONE cross-architecture scatter: colour = editor/reference (§4 colours), marker shape = world model.
ax = axes[1]
MARK7 = {"GRU": "o", "RSSM": "s"}
for m in MARK7:
    for n in REF_ORDER + ED_ORDER:
        ax.scatter(METRICS[m][n]["readout"], METRICS[m][n]["nextstep"], s=90, color=COL[n],
                   marker=MARK7[m], edgecolor="k", linewidth=0.8, zorder=3)
ax.set_xscale("symlog", linthresh=2.0)
ax.set_xlabel("readout RMSE at edit step (position units; axis linear ≤ 2, log > 2) — lower is better")
ax.set_ylabel("GT next-step RMSE (obs intensity)\n— lower is better")
ax.set_title("(b) editing head-to-head: state-space vs observation-space accuracy, both models")
style_ax(ax)
handles7 = [Patch(facecolor=COL[n], edgecolor="k", label=n) for n in REF_ORDER + ED_ORDER]
handles7 += [Line2D([0], [0], marker=MARK7[m], linestyle="none", markersize=8,
                    markerfacecolor="0.88", markeredgecolor="k", label=m) for m in MARK7]
ax.legend(handles=handles7, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False,
          title="colour = editor / reference\nshape = world model", title_fontsize=8)

fig.suptitle("Fig 7 — Summary", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(f"{OUT}/fig7_summary.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

# plain-text mirror (agent-readable / no-render)
print("Fig 7b points — (readout RMSE, GT next-step RMSE) per world model and editor/reference:")
for m in MARK7:
    print(f"  {m}: " + " | ".join(f"{n} ({METRICS[m][n]['readout']:.2f}, {METRICS[m][n]['nextstep']:.3f})"
                                  for n in REF_ORDER + ED_ORDER))

In [ ]:
# [26] §5 — consolidated summary: every headline number, both models, in clearly-demarcated tables + PNG manifest.
from IPython.display import Markdown, display

display(Markdown(
    "## Master editability — consolidated summary (GRU + refined RSSM)\n"
    "**Current results (updated 2026-07-15).** Every number below is recomputed in this notebook run; "
    "metric formulas live in the definitions table (after the bootstrap) and the §4 definitions table."))

# ---- §1 geometry ----
display(Markdown("**§1 Geometry** — dimensionality and curvature of the visited-state manifold. "
                 "Physical reference: 8 degrees of freedom (2 objects × (2 pos + 2 vel))."))
geo_sum = {
    "GRU":  dict(hull90=dims_g[0.90], twonn=twonn_g, mle=mle_g, tangent=CITED["tangent_angle_gru"]),
    "RSSM": dict(hull90=dims_r[0.90], twonn=twonn_r, mle=mle_r, tangent=CITED["tangent_angle_rssm"]),
}
GEO_SUM_COLS = [("hull90", "PCA hull dim @90% var", "{:d}"), ("twonn", "intrinsic dim (TwoNN)", "{:.1f}"),
                ("mle", "intrinsic dim (MLE k=20)", "{:.1f}"),
                ("tangent", "tangent rotation @ NN spacing (deg, cited)", "{:.0f}")]
display(md_table(geo_sum, GEO_SUM_COLS, row_hdr="model"))

# ---- §2 recoverability ----
display(Markdown("**§2 Recoverability** — probe R² for reading the physical state out of a single hidden state "
                 "(late-t = frames t ≥ 15, the converged-filter regime; all probes in-sample)."))
rec_sum = {}
for lbl in ["GRU", "RSSM"]:
    o = vel_res[(lbl, "late")]
    rec_sum[lbl] = dict(pos_lin=pos_res[(lbl, "lin")]["r2"], pos_mlp=pos_res[(lbl, "mlp")]["r2"],
                        vel_lin=o[("sf", "lin")]["r2"], vel_mlp=o[("sf", "mlp")]["r2"],
                        vel_gain=o[("win", "mlp")]["r2"] - o[("sf", "mlp")]["r2"])
REC_SUM_COLS = [("pos_lin", "position R² (linear)", "{:.2f}"), ("pos_mlp", "position R² (MLP)", "{:.2f}"),
                ("vel_lin", "velocity R² (single-frame linear, late-t)", "{:.2f}"),
                ("vel_mlp", "velocity R² (single-frame MLP, late-t)", "{:.2f}"),
                ("vel_gain", "two-frame − single-frame (MLP, late-t)", "{:+.3f}")]
display(md_table(rec_sum, REC_SUM_COLS, row_hdr="model"))
display(Markdown("_Reading velocity needs a nonlinear probe but only one hidden state (the two-frame gain is "
                 "≈ 0): the readout is instantaneous, not temporal._"))

# ---- §3 canonicality ----
display(Markdown("**§3 Canonicality** — fiber residual ‖block − g(pos,vel)‖ / ‖block‖ "
                 "(0 = the block is fully a function of the physical state)."))
fib_sum = {name: dict(lin=fiber[name]["lin"][0], mlp=fiber[name]["mlp"][0], r2=fiber[name]["mlp"][1])
           for name in ["GRU h (256)", "RSSM h_det (256)", "RSSM full (320)", "RSSM s_stoch (64)"]}
FIB_SUM_COLS = [("lin", "linear g: residual fraction", "{:.3f}"), ("mlp", "MLP g: residual fraction", "{:.3f}"),
                ("r2", "MLP g: R² on block", "{:.3f}")]
display(md_table(fib_sum, FIB_SUM_COLS, row_hdr="state block"))

# ---- §4 editing head-to-head ----
display(Markdown("**§4 Editing head-to-head** — same edit set on both world models; references above the "
                 "editors. The true-state swap is the upper bound any hidden-state editor could reach."))
SUM4_COLS = [("readout", "readout RMSE (position)", "{:.3f}"), ("nextstep", "GT next-step RMSE (obs)", "{:.3f}"),
             ("pct_swap", "obs-change (% of swap)", "{:.1f}"), ("ghost", "ghost-ray ratio", "{:.3f}"),
             ("loo_resid", "leave-out local-PCA residual (fraction)", "{:.2f}")]
for m in ["GRU", "RSSM"]:
    display(Markdown(f"_{m} reference scales: true-state-swap obs-change {SWAP_CHG[m]:.3f} (the 100% "
                     f"denominator); real-state leave-out local-PCA residual {REAL_LOO[m]:.2f}; unsteered GT "
                     f"next-step RMSE {METRICS[m]['Unsteered']['nextstep']:.3f}._"))
    display(md_table({n: METRICS[m][n] for n in REF_ORDER + ED_ORDER}, SUM4_COLS, row_hdr=f"{m} — state"))

# ---- §4 rollout behaviour + geodesic budget ----
display(Markdown("**§4 rollout behaviour and budget check** — the decoder-gradient rollout **collapses "
                 "off-distribution** (its GT-trajectory RMSE climbs back to the no-edit level while its "
                 "distance to the unsteered rollout stays large — degeneration, not a return to the no-edit "
                 "trajectory); the PCA-geodesic readout **asymptotes** (the remaining readout gap does not "
                 "close with a 5× iteration budget)."))
beh_sum = {}
for m in ["GRU", "RSSM"]:
    beh_sum[m] = dict(dg0=STEP_RMSE[m]["Decoder gradient"][0], dg4=STEP_RMSE[m]["Decoder gradient"][4],
                      uns4=STEP_RMSE[m]["Unsteered"][4], du4=DIST_TO_UNSTEERED[m]["Decoder gradient"][4],
                      geo120=summary_rows[m]["rmse_at_iter120"], geoF=summary_rows[m]["rmse_final"])
BEH_SUM_COLS = [("dg0", "decoder-gradient GT-trajectory RMSE @ step 0", "{:.3f}"),
                ("dg4", "same @ step 4", "{:.3f}"), ("uns4", "unsteered @ step 4 (reference)", "{:.3f}"),
                ("du4", "decoder-gradient distance to unsteered @ step 4", "{:.3f}"),
                ("geo120", "PCA-geodesic readout RMSE @ iteration 120 (32 samples)", "{:.3f}"),
                ("geoF", "same @ the K=600 plateau", "{:.3f}")]
display(md_table(beh_sum, BEH_SUM_COLS, row_hdr="model"))

# ---- figure manifest ----
png_paths = sorted(os.path.join(OUT, f) for f in os.listdir(OUT) if f.endswith(".png"))
display(Markdown("**Figure PNG manifest** (regenerated by this run):\n\n" + "\n".join(f"- `{p}`" for p in png_paths)))
print(f"{len(png_paths)} PNGs in {OUT}")